## Script for histograms creation

In [37]:
%reload_ext autoreload
%autoreload 2

import glob
import json
import time
import os
import hashlib
import yaml


import ROOT
import utils.physics_utils as PU
from utils.general_utils import get_valid_files_fast, log_analysis_run, build_cut_expression
from utils.hist_utils import (
    split_variable_group,
    fill_1d_histograms,
    fill_2d_histograms,
    draw_and_save_1d_histograms,
    draw_and_save_2d_histograms,
)

ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gSystem.Load("/workspace/KLOE/Subanalysis/CPFit/libCPFit.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/Codes/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/FortranAnalysis/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Subanalysis/CPFit/inc")

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/FortranAnalysis/inc/fort_common.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/const.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/HypothesisCodes.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/SelectionCodes.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/charged_mom.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/kloe_class.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Subanalysis/CPFit/inc/cpfit.hpp"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

# Enrichment for control samples
channTitle["has000"] = "Event includes K_{S}#rightarrow#pi^{+}#pi^{-}"
channTitle["!has000"] = "Event does not include K_{S}#rightarrow#pi^{+}#pi^{-}"
  
channColor["has000"] = ROOT.kRed
channColor["!has000"] = ROOT.kBlue

print(channName)

{0: 'Data', 1: 'Signal', 2: 'Regeneration', 3: 'Omega', 4: '3pi0', 5: 'Semileptonic', 6: 'Other', 7: 'pi+pi-pi+pi-', 8: 'MC sum'}


In [70]:
base_path = "/workspace/root_notebooks/results"

################ SETTINGS ##################################
analysis_dir = "kl_pm_control_sample"
img_dir = "img"
analysis_hash = "7b6c187f3e4b0390d51437281fd4292b"
root_file_name = "signal_analysis_chann_div_1D_variables.root"
mode = "full"
nocut_hash = "eff9a201"
log_flag = False
draw_data_mc_sum = True

if analysis_dir == "kl_pm_control_sample":
  root_file_name_base = "tag_ks_pm"
  delta_t_vars = ["Delta_t_fit",
                  "Delta_t_rec"]
  signal_channel = "pi+pi-pi+pi-"
elif analysis_dir == "ks_pm_control_sample":
  root_file_name_base = "tag_kl_000"
  delta_t_vars = ["Delta_t_fit",
                  "Delta_t_tri_fit",
                  "Delta_t_rec"]
  signal_channel = "3pi0"
elif analysis_dir == "ks_00_control_sample":
  root_file_name_base = "tag_kl_Semileptonic"
  delta_t_vars = ["Delta_t_fit",
                  "Delta_t_tri_fit",
                  "Delta_t_rec"]
  signal_channel = "Semileptonic"
elif analysis_dir == "signal_analysis_cut_checks":
  root_file_name_base = "signal_analysis"
  delta_t_vars = ["Delta_t_fit",
                  "Delta_t_tri_fit",
                  "Delta_t_rec"]
  signal_channel = "Signal"

branching_ratio_mode = "ks_pm"
branching_ratio = 1.0

# if branching_ratio_mode == "ks_pm":
#     branching_ratio = 0.6920
# elif branching_ratio_mode == "ks_00":
#     branching_ratio = 0.3069

division = "channels"

if division == "tag":
  root_file_name = f"{root_file_name_base}_1D_variables.root"
  if analysis_dir == "kl_pm_control_sample":
      mc_flags = {"hasKsPM": "hasKsPM", "!hasKsPM": "!hasKsPM", "MC sum": "MC sum", "Data": "Data"}
  elif analysis_dir == "ks_pm_control_sample":
      mc_flags = {"has000": "has000", "!has000": "!has000", "MC sum": "MC sum", "Data": "Data"}
  elif analysis_dir == "ks_00_control_sample":
      mc_flags = {"hasSemileptonic": "hasSemileptonic", "!hasSemileptonic": "!hasSemileptonic", "MC sum": "MC sum", "Data": "Data"}
else:
    root_file_name = f"{root_file_name_base}_chann_div_1D_variables.root"
    mc_flags = channName
############################################################

hist_file_path = f"{base_path}/{analysis_dir}/{analysis_hash}/{root_file_name}"

#################### ANALYSIS REGISTRY #####################
import pandas as pd

from utils.general_utils import load_2d_config, load_fiducial_filters
analysis_registry = f"{base_path}/{analysis_dir}/{analysis_hash}/analysis_registry.csv"
df = pd.read_csv(analysis_registry)

hash_available = df["hash"]

########################## LOAD HISTOGRAM CONFIGURATION ##############################
hist_config_file_path = f"{base_path}/{analysis_dir}/config/histograms_config_1D.yaml"
with open(hist_config_file_path, "r") as f:
    hist_config = yaml.safe_load(f)

hist_config_2D_file_path = f"{base_path}/{analysis_dir}/config/histograms_config_2D.yaml"
with open(hist_config_2D_file_path, "r") as f:
    hist_config_2D = yaml.safe_load(f)
variables_2D, modes_2D, TRUTH_ONLY_VARS_2D, TRUTH_MCTRUTH_ID_2D = load_2d_config(
    hist_config_2D, hist_config
)
filters_1D, filters_2D = load_fiducial_filters(hist_config, hist_config_2D)
#####################################################################################  
print(hist_config_2D)

{'truth_mctruth_id_2d': 7, 'truth_only_vars_2d': ['Delta_t_rec_Delta_t_MC', 'Delta_t_MC_Pmiss', 'Delta_t_MC_Delta_t_rec_resolution', 'Delta_t_MC_Delta_t_fit_resolution'], 'histograms_2d': {'Delta_t_rec_Delta_t_MC': {'x': 'Delta_t_rec', 'y': 'Delta_t_MC', 'modes': {'full': {'x': 'full', 'y': 'full'}, 'zoom': {'x': 'zoom', 'y': 'zoom'}}}, 'KchrecKS_5_KchrecKL_5': {'x': 'KchrecKS_5', 'y': 'KchrecKL_5', 'modes': {'full': {'x': 'full', 'y': 'full'}, 'zoom': {'x': 'zoom', 'y': 'zoom'}}}, 'Emiss_Pmiss': {'x': 'Emiss', 'y': 'Pmiss', 'modes': {'full': {'x': 'full', 'y': 'full'}, 'zoom': {'x': 'zoom', 'y': 'zoom'}}}, 'rho_pm_closest_rho_00_closest': {'x': 'rho_pm_closest', 'y': 'rho_00_closest', 'fiducial_volume': 'omega', 'modes': {'full': {'x': 'full', 'y': 'full'}}}, 'cos_trk1_K_cos_trk2_K': {'x': 'cos_trk1_K', 'y': 'cos_trk2_K', 'fiducial_volume': 'close_tracks', 'modes': {'full': {'x': 'full', 'y': 'full'}}}, 'Delta_t_fit_Pmiss': {'x': 'Delta_t_fit', 'y': 'Pmiss', 'modes': {'full': {'x': 'f

## Making 1D histograms

In [66]:
try:
  variables_1D = list(hist_config.get("histograms_1d", {}).keys())
  TRUTH_ONLY_VARS_1D = hist_config.get("truth_only_vars_1d", [])
  TRUTH_MCTRUTH_ID_1D = hist_config.get("truth_mctruth_id_1d", 1)

  variables_1D_titles = []
  unit_1D = []
  range_X_1D = {}
  modes = {}

  for var in variables_1D:
      variables_1D_titles.append(hist_config["histograms_1d"][var].get("title", ""))
      unit_1D.append(hist_config["histograms_1d"][var].get("unit", ""))
      modes[var] = list(hist_config["histograms_1d"][var].get("range_x", {}).keys())

      range_X_1D[var] = {}
      for mode_tmp in modes[var]:
          range_X_1D[var][mode_tmp] = hist_config["histograms_1d"][var].get("range_x", {}).get(mode_tmp, [])

  print(variables_1D_titles)
except Exception as e:
  print(f"Error occurred: {e}")

['Missing energy', 'Missing momentum', 'Missing invariant mass', 'm^{inv}_{+-,1} - m_{K^{0}}', 'm^{inv}_{+-,2} - m_{K^{0}}', '#rho_{K_{1}}', '#rho_{K_{2}}', 'z_{K_{1}} - z_{IP}', 'z_{K_{2}} - z_{IP}', 'R_{K_{1}}', 'R_{K_{2}}', '|#vec{p}^{CM}_{+,1} + #vec{p}^{CM}_{-,1}| - |p^{CM}_{K}|', '|#vec{p}^{CM}_{+,2} + #vec{p}^{CM}_{-,2}| - |p^{CM}_{K}|', 'Q_{miss}^{K_{1}}', 'Q_{miss}^{K_{2}}', '#Deltat_{MC}', '#Deltat_{fit}', '#Deltat_{rec}', 'Number of neutral clusters', '#Deltat_{fit} - #Deltat_{MC}', '#Deltat_{rec} - #Deltat_{MC}', 'm^{inv}_{+-,1} - m_{K^{0}}', 'm^{inv}_{+-,2} - m_{K^{0}}', 'R_{K_{1}} - R_{K_{1}}^{MC}', 'R_{K_{2}} - R_{K_{2}}^{MC}', '#rho_{K_{1}} - #rho_{K_{1}}^{MC}', '#rho_{K_{2}} - #rho_{K_{2}}^{MC}', '(z_{K_{1}} - z_{IP}) - (z_{K_{1}} - z_{IP})^{MC}', '(z_{K_{2}} - z_{IP}) - (z_{K_{2}} - z_{IP})^{MC}', 'x_{IP}^{KS} - x_{IP}^{KL}', 'y_{IP}^{KS} - y_{IP}^{KL}', 'z_{IP}^{KS} - z_{IP}^{KL}', 'p_{x}^{K_{1}} - p_{x}^{K_{1}, MC}', 'p_{y}^{K_{1}} - p_{y}^{K_{1}, MC}', 'p_{z}^{K_{1

In [67]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(hist_file_path)
ROOT.TH1.AddDirectory(False)

#################### OBJECT TO FIND ################################
histograms = {}

print(f"Hash available: {hash_available}")

for hash in hash_available:
    if hash not in [key.GetName() for key in hist_file.GetListOfKeys()]:
        print(f"Hash {hash} not found in file.")
        continue
    
    histograms[hash] = {}

    dir = hist_file.Get(hash)

    for i, var in enumerate(variables_1D):
      dir_obj = hist_file.Get(f"{hash}/{var}")

      if not isinstance(dir_obj, ROOT.TDirectory):
          continue

      if var not in histograms[hash]:
          histograms[hash][var] = {}

      for chann in mc_flags.values():
        hist_name = f"{var}_{chann}_{mode}"

        if hist_name not in [key.GetName() for key in dir_obj.GetListOfKeys()]:
            continue
        histograms[hash][var][chann] = dir_obj.Get(hist_name).Clone()

hist_file.Close()


Hash available: 0      nocuts
1    21f528cb
2    eff9a201
3    0920bc7b
4    ed9f9783
5    11f92bec
6    f63b399e
Name: hash, dtype: object


In [68]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{analysis_hash}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)

################## DRAW USING THSTACK ###########################
canvases = []
legends = []
stacks = []

for hash in histograms:    
    path_hash = f"{img_dir_path}/{hash}"
    os.makedirs(path_hash, exist_ok=True)

    root_file_name = f"{path_hash}/histograms.root"
    root_file = ROOT.TFile(root_file_name, "UPDATE")


    for i, var in enumerate(histograms[hash]):
        path_var = f"{path_hash}/{var}"
        os.makedirs(path_var, exist_ok=True)

        root_file.mkdir(var)
        root_file.cd(var)
        
        legend = ROOT.TLegend(0.15, 0.75, 0.84875, 0.9)
        legend.SetTextFont(62)
        legend.SetTextSize(0.025)
        legend.SetMargin(0.1)
        legend.SetNColumns(2)
        
        # Dodajemy hash do nazwy, aby uniknąć konfliktów w pamięci ROOTa przy wielu folderach
        stack = ROOT.THStack(f"{var}_stack_{hash}", "")
        
        for flag in histograms[hash][var]:
            hist = histograms[hash][var][flag]

            if not draw_data_mc_sum and (flag == "Data" or flag == "MC sum"):
                continue

            hist.SetMaximum(hist.GetBinContent(hist.GetMaximumBin()) * 1.3)

            if hist:
                hist.SetLineColor(channColor[flag])

                if log_flag:
                    hist.SetMinimum(0.1)
                
                if flag != "Data":
                    # Opcję rysowania przekazujemy bezpośrednio w metodzie Add dla THStack
                    stack.Add(hist, "hist")
                    legend.AddEntry(hist, channTitle[flag], "l")
                else:
                    hist.SetMarkerStyle(20)
                    hist.SetMarkerColor(channColor[flag])
                    stack.Add(hist, "pe1")
                    legend.AddEntry(hist, channTitle[flag], "pe1")

                hist.Write(f"{var}_{flag}_{mode}", ROOT.TObject.kOverwrite)
                            
        canvas = ROOT.TCanvas(f"{var}_canvas_{hash}", f"{var} Canvas", 600, 600)
        
        stack.Draw("NOSTACK")
        
        if stack.GetHistogram():
            resolution = stack.GetHistogram().GetXaxis().GetBinWidth(1)
            
            stack.GetHistogram().GetXaxis().SetTitle(f"{variables_1D_titles[variables_1D.index(var)]} [{unit_1D[variables_1D.index(var)]}]")
            stack.GetHistogram().GetYaxis().SetTitle(f"Counts / {resolution:.2f} {unit_1D[variables_1D.index(var)]}")

        ####### Set log y ########
        if log_flag:
          stack.SetMinimum(0.1)
          stack.SetMaximum(stack.GetMaximum() * 50)
        else:
          stack.SetMinimum(0)
                
        canvas.SetLogy(log_flag)
        ##########################

        legend.Draw()

        canvas.Draw()
        canvas.SaveAs(f"{path_var}/{var}_{mode}_logy_{log_flag}.pdf")
        canvas.Write(f"{var}_{mode}_logy_{log_flag}", ROOT.TObject.kOverwrite)
        canvas.Close()
    root_file.Close()

Error in <TFile::mkdir>: An object with name Emiss exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/nocuts/Emiss/Emiss_full_logy_False.pdf has been created
Error in <TFile::mkdir>: An object with name Pmiss exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/nocuts/Pmiss/Pmiss_full_logy_False.pdf has been created
Error in <TFile::mkdir>: An object with name MissDiff exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/nocuts/MissDiff/MissDiff_full_logy_False.pdf has been created
Error in <TFile::mkdir>: An object with name KchrecKS_5 exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/nocuts/KchrecKS_5/KchrecKS_5_full_logy_False.p

## Making 2D histograms

In [69]:
# Ustawienia 2D: nazwy plików można zmienić niezależnie od pliku 1D.
input_name_1d = os.path.basename(hist_file_path)
if not input_name_1d.endswith('_1D_variables.root'):
    raise ValueError('Ustaw pliki 2D: nazwa pliku 1D musi kończyć się na _1D_variables.root')
prefix_2d = input_name_1d[:-len('_1D_variables.root')]
root_file_2d = f'{prefix_2d}_2D_variables.root'
root_file_2d_truth = f'{prefix_2d}_2D_truth_variables.root'
mode_2d = mode
logz_2d = False
draw_truth_2d = True

result_dir_2d = os.path.join(base_path, analysis_dir, analysis_hash)
var_keys_2d = ['_'.join(pair) for pair in variables_2D]
truth_keys_2d = set(TRUTH_ONLY_VARS_2D)
sources_2d = [(root_file_2d, [v for v in var_keys_2d if v not in truth_keys_2d], '2D')]
if draw_truth_2d and truth_keys_2d:
    sources_2d.append((root_file_2d_truth, [v for v in var_keys_2d if v in truth_keys_2d], '2D_truth'))

for filename_2d, variables_for_file, kind in sources_2d:
    input_path_2d = os.path.join(result_dir_2d, filename_2d)
    input_file_2d = ROOT.TFile.Open(input_path_2d, 'READ')
    if not input_file_2d or input_file_2d.IsZombie():
        raise OSError(f'Nie można otworzyć pliku ROOT: {input_path_2d}')
    try:
        for cut_hash_2d in hash_available:
            if not input_file_2d.GetDirectory(str(cut_hash_2d)):
                print(f'Hash {cut_hash_2d} nie występuje w {filename_2d}')
                continue
            output_dir_2d = os.path.join(result_dir_2d, img_dir, str(cut_hash_2d))
            os.makedirs(output_dir_2d, exist_ok=True)
            output_path_2d = os.path.join(output_dir_2d, f'histograms_{kind}.root')
            output_file_2d = ROOT.TFile.Open(output_path_2d, 'UPDATE')
            if not output_file_2d or output_file_2d.IsZombie():
                raise OSError(f'Nie można otworzyć pliku ROOT: {output_path_2d}')
            try:
                for var_2d in variables_for_file:
                    if mode_2d not in modes_2D[var_2d]:
                        continue
                    source_dir_2d = input_file_2d.GetDirectory(f'{cut_hash_2d}/{var_2d}')
                    if not source_dir_2d:
                        print(f'Brak {cut_hash_2d}/{var_2d} w {filename_2d}')
                        continue
                    prefix_key_2d = f'{var_2d}_{mode_2d}_'
                    keys_2d = dict.fromkeys(key.GetName() for key in source_dir_2d.GetListOfKeys())
                    matching_2d = [key for key in keys_2d if key.startswith(prefix_key_2d)]
                    if not matching_2d:
                        print(f'Brak trybu {mode_2d} dla {cut_hash_2d}/{var_2d} w {filename_2d}')
                    for key_2d in matching_2d:
                        channel_2d = key_2d[len(prefix_key_2d):]
                        if not draw_data_mc_sum and channel_2d in ('Data', 'MC sum'):
                            continue
                        hist_2d = source_dir_2d.Get(key_2d)
                        if not hist_2d or not hist_2d.InheritsFrom('TH2'):
                            raise TypeError(f'{cut_hash_2d}/{var_2d}/{key_2d} nie jest histogramem 2D')
                        path_var_2d = os.path.join(output_dir_2d, var_2d)
                        os.makedirs(path_var_2d, exist_ok=True)
                        target_dir_2d = output_file_2d.GetDirectory(var_2d) or output_file_2d.mkdir(var_2d)
                        target_dir_2d.cd()
                        hist_2d.Write(key_2d, ROOT.TObject.kOverwrite)
                        canvas_2d = ROOT.TCanvas(f'{key_2d}_{cut_hash_2d}_canvas', channel_2d, 600, 600)
                        canvas_2d.SetLogz(logz_2d)
                        hist_2d.Draw('COLZ')
                        canvas_2d.Draw()
                        canvas_2d.SaveAs(os.path.join(path_var_2d, f'{key_2d}_logz_{logz_2d}.pdf'))
                        canvas_2d.Write(f'{key_2d}_logz_{logz_2d}', ROOT.TObject.kOverwrite)
                        canvas_2d.Close()
            finally:
                output_file_2d.Close()
    finally:
        input_file_2d.Close()

Brak nocuts/Delta_t_fit_Pmiss w tag_ks_pm_chann_div_2D_variables.root
Brak nocuts/Delta_t_rec_Pmiss w tag_ks_pm_chann_div_2D_variables.root
Brak 21f528cb/Delta_t_fit_Pmiss w tag_ks_pm_chann_div_2D_variables.root
Brak 21f528cb/Delta_t_rec_Pmiss w tag_ks_pm_chann_div_2D_variables.root
Brak eff9a201/Delta_t_fit_Pmiss w tag_ks_pm_chann_div_2D_variables.root
Brak eff9a201/Delta_t_rec_Pmiss w tag_ks_pm_chann_div_2D_variables.root
Brak 0920bc7b/Delta_t_fit_Pmiss w tag_ks_pm_chann_div_2D_variables.root
Brak 0920bc7b/Delta_t_rec_Pmiss w tag_ks_pm_chann_div_2D_variables.root
Brak ed9f9783/Delta_t_fit_Pmiss w tag_ks_pm_chann_div_2D_variables.root
Brak ed9f9783/Delta_t_rec_Pmiss w tag_ks_pm_chann_div_2D_variables.root
Brak f63b399e/Delta_t_fit_Pmiss w tag_ks_pm_chann_div_2D_variables.root
Brak f63b399e/Delta_t_rec_Pmiss w tag_ks_pm_chann_div_2D_variables.root
Brak nocuts/Delta_t_MC_Pmiss w tag_ks_pm_chann_div_2D_truth_variables.root
Brak nocuts/Delta_t_MC_Delta_t_rec_resolution w tag_ks_pm_chann_d

Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/nocuts/KchrecKS_5_KchrecKL_5/KchrecKS_5_KchrecKL_5_full_Data_logz_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/nocuts/KchrecKS_5_KchrecKL_5/KchrecKS_5_KchrecKL_5_full_Signal_logz_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/nocuts/KchrecKS_5_KchrecKL_5/KchrecKS_5_KchrecKL_5_full_Regeneration_logz_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/nocuts/KchrecKS_5_KchrecKL_5/KchrecKS_5_KchrecKL_5_full_Omega_logz_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51

### Check of additional cut efficiency + purity

In [71]:
cut_hash = "11f92bec"
var = "Pmiss"
cut_value = 10.0

tmp_histograms = {signal_channel: histograms[cut_hash][var][signal_channel],
                  "MC sum": histograms[cut_hash][var]["MC sum"]}

counts_initial = {signal_channel: tmp_histograms[signal_channel].Integral(0, tmp_histograms[signal_channel].GetXaxis().GetNbins() + 1),
                  "MC sum": tmp_histograms["MC sum"].Integral(0, tmp_histograms["MC sum"].GetXaxis().GetNbins() + 1)}


cut_bin = tmp_histograms[signal_channel].GetXaxis().FindBin(cut_value)

counts_cut = {signal_channel: tmp_histograms[signal_channel].Integral(0, cut_bin - 1),
              "MC sum": tmp_histograms["MC sum"].Integral(0, cut_bin - 1)}

print("Initial counts:", counts_initial)
print("Counts after cut:", counts_cut)
print("Efficiency:", {key: counts_cut[key] / counts_initial[key] for key in counts_initial})
print("Purity:", {key: counts_cut[key] / counts_cut["MC sum"] for key in counts_cut})

Initial counts: {'pi+pi-pi+pi-': 238921.96252362948, 'MC sum': 1938239.419706406}
Counts after cut: {'pi+pi-pi+pi-': 226436.83679593084, 'MC sum': 244143.1384938268}
Efficiency: {'pi+pi-pi+pi-': 0.9477439177385635, 'MC sum': 0.12596129044306006}
Purity: {'pi+pi-pi+pi-': 0.9274757349023608, 'MC sum': 1.0}


## Check of purity in the function of $\Delta t$

In [72]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{analysis_hash}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)

signal = signal_channel
sum = "MC sum"

delta_t_signal = {}
delta_t_sum = {}

for hash in histograms:
    delta_t_signal[hash] = []
    delta_t_sum[hash] = []
    for var in delta_t_vars:
        if var not in histograms[hash]:
            continue

        delta_t_signal[hash].append(histograms[hash][var][signal])
        delta_t_sum[hash].append(histograms[hash][var][sum])


######################### DIVIDE #############################
delta_t_divided = {}

for hash in delta_t_signal:
    delta_t_divided[hash] = []
    for signal_hist, sum_hist in zip(delta_t_signal[hash], delta_t_sum[hash]):
        hist_temp = signal_hist.Clone()
        hist_temp.Divide(sum_hist)
        delta_t_divided[hash].append(hist_temp)


######################## PLOT ###########################
canvases = []

ROOT.gStyle.SetOptStat(0)

for hash in delta_t_divided:
    path_hash = f"{img_dir_path}/{hash}/efficiency_purity_plots"
    os.makedirs(path_hash, exist_ok=True)

    root_file_name = f"{path_hash}/eff_pur_corr_factor.root"
    root_file = ROOT.TFile(root_file_name, "UPDATE")

    root_file.mkdir("purity")
    for hist, title in zip(delta_t_divided[hash], delta_t_vars):
        c = ROOT.TCanvas("")
        hist.SetMarkerStyle(ROOT.kFullCircle)
        hist.SetLineColor(ROOT.kBlack)
        hist.GetYaxis().SetRangeUser(0, 1.1)
        hist.GetYaxis().SetTitle("Purity")
        hist.Draw("PE1")
        root_file.cd("purity")
        hist.Write(f"purity_{title.replace(' ', '_')}_per_channel", ROOT.TObject.kOverwrite)
        c.Draw()
        title = f"purity_{title.replace(' ', '_')}_per_channel.pdf"
        c.SaveAs(f"{path_hash}/{title}")
        c.Write(title, ROOT.TObject.kOverwrite)
        c.Close()
    root_file.Close()

        

Error in <TFile::mkdir>: An object with name purity exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/nocuts/efficiency_purity_plots/purity_Delta_t_fit_per_channel.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/nocuts/efficiency_purity_plots/purity_Delta_t_rec_per_channel.pdf has been created
Error in <TFile::mkdir>: An object with name purity exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/21f528cb/efficiency_purity_plots/purity_Delta_t_fit_per_channel.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/21f528cb/efficiency_purity_plots/purity_Delta_t_rec_per_channel.pdf has been created
Error in <TFile::mkdir

## Check of efficiency in the function of $\Delta t$

In [73]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{analysis_hash}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)

mode_eff = "subtract_bkg"

if mode_eff == "subtract_bkg":
  signal = signal_channel
  sum = signal_channel
else:
  signal = "MC sum"
  sum = "MC sum"

delta_t_signal = {}
delta_t_sum = {}

for hash in histograms:
    delta_t_signal[hash] = []
    delta_t_sum[hash] = []
    for var in delta_t_vars:
        if var not in histograms[hash]:
            continue

        delta_t_signal[hash].append(histograms[hash][var][signal])
        delta_t_sum[hash].append(histograms[nocut_hash][var][sum])


######################### DIVIDE #############################
delta_t_divided_signal = {}

for hash in delta_t_signal:
    delta_t_divided_signal[hash] = []
    for signal_hist, sum_hist in zip(delta_t_signal[hash], delta_t_sum[hash]):
        hist_temp = signal_hist.Clone()
        if mode_eff != "subtract_bkg":
            sum_hist.Scale(branching_ratio)
        hist_temp.Divide(sum_hist)
        delta_t_divided_signal[hash].append(hist_temp)


######################## PLOT ###########################
canvases = []

ROOT.gStyle.SetOptStat(0)

for hash in delta_t_divided_signal:
    path_hash = f"{img_dir_path}/{hash}/efficiency_purity_plots"
    os.makedirs(path_hash, exist_ok=True)
    root_file_name = f"{path_hash}/eff_pur_corr_factor.root"
    root_file = ROOT.TFile(root_file_name, "UPDATE")

    root_file.mkdir("efficiency_MC")
    root_file.cd("efficiency_MC")
    for hist, title in zip(delta_t_divided_signal[hash], delta_t_vars):
        c = ROOT.TCanvas("")
        hist.SetMarkerStyle(ROOT.kFullCircle)
        hist.SetLineColor(ROOT.kBlack)
        hist.GetYaxis().SetRangeUser(0, 1.1)
        hist.GetYaxis().SetTitle("Efficiency")
        hist.Draw("PE1")
        if mode_eff == "subtract_bkg":
          hist.Write(f"efficiency_{title.replace(' ', '_')}_MC_subtract_bkg", ROOT.TObject.kOverwrite)
        else:
          hist.Write(f"efficiency_{title.replace(' ', '_')}_MC", ROOT.TObject.kOverwrite)
        c.Draw()

        if mode_eff == "subtract_bkg":
          title = f"efficiency_{title.replace(' ', '_')}_MC_subtract_bkg.pdf"
        else:
          title = f"efficiency_{title.replace(' ', '_')}_MC.pdf"
        c.SaveAs(f"{path_hash}/{title}")
        c.Write(title, ROOT.TObject.kOverwrite)
        c.Close()
    root_file.Close()
  

Error in <TFile::mkdir>: An object with name efficiency_MC exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/nocuts/efficiency_purity_plots/efficiency_Delta_t_fit_MC_subtract_bkg.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/nocuts/efficiency_purity_plots/efficiency_Delta_t_rec_MC_subtract_bkg.pdf has been created
Error in <TFile::mkdir>: An object with name efficiency_MC exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/21f528cb/efficiency_purity_plots/efficiency_Delta_t_fit_MC_subtract_bkg.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/21f528cb/efficiency_purity_plots/efficiency_Delta_t_rec_MC_subtract_b

In [14]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{analysis_hash}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)

if mode_eff == "subtract_bkg":
  signal = "Data"
  signal_MC = signal_channel
  sum = "MC sum"
else:
  signal = "Data"
  signal_MC = ""
  sum = "MC sum"

delta_t_signal = {}
delta_t_sum = {}

for hash in histograms:
    delta_t_signal[hash] = []
    delta_t_sum[hash] = []
    for var in delta_t_vars:
        if var not in histograms[hash]:
            continue

        hist_data_tmp = histograms[hash][var][signal].Clone(f"{hash}_{var}_{signal}")
        hist_data_nocut_tmp = histograms[nocut_hash][var][signal].Clone(f"{nocut_hash}_{var}_{signal}")

        if mode_eff == "subtract_bkg":
          for key in histograms[hash][var]:
              if key == signal or key == signal_MC or key == sum:
                continue

              hist_data_tmp.Add(histograms[hash][var][key], -1.0)
              hist_data_nocut_tmp.Add(histograms[nocut_hash][var][key], -1.0)

        else:
          hist_data_nocut_tmp.Scale(branching_ratio)
              
        delta_t_signal[hash].append(hist_data_tmp)
        delta_t_sum[hash].append(hist_data_nocut_tmp)


######################### DIVIDE #############################
delta_t_divided_data = {}

for hash in delta_t_signal:
    delta_t_divided_data[hash] = []
    for signal_hist, sum_hist in zip(delta_t_signal[hash], delta_t_sum[hash]):
        hist_temp = signal_hist.Clone()
        hist_temp.Sumw2()
        hist_temp.Divide(sum_hist)
        delta_t_divided_data[hash].append(hist_temp)


######################## PLOT ###########################
canvases = []

ROOT.gStyle.SetOptStat(0)

for hash in delta_t_divided_data:
    path_hash = f"{img_dir_path}/{hash}/efficiency_purity_plots"
    os.makedirs(path_hash, exist_ok=True)

    root_file_name = f"{path_hash}/eff_pur_corr_factor.root"
    root_file = ROOT.TFile(root_file_name, "UPDATE")
    root_file.mkdir("efficiency_data")
    root_file.cd("efficiency_data")
    for hist, title in zip(delta_t_divided_data[hash], delta_t_vars):
        c = ROOT.TCanvas("")
        hist.SetMarkerStyle(ROOT.kFullCircle)
        hist.SetLineColor(ROOT.kBlack)
        hist.GetYaxis().SetRangeUser(0, 1.1)
        hist.GetYaxis().SetTitle("Efficiency")
        hist.Draw("PE1")
        if mode_eff == "subtract_bkg":
          hist.Write(f"efficiency_{title.replace(' ', '_')}_data_subtract_bkg", ROOT.TObject.kOverwrite)
        else:
          hist.Write(f"efficiency_{title.replace(' ', '_')}_data", ROOT.TObject.kOverwrite)
        c.Draw()
        if mode_eff == "subtract_bkg":
            title = f"efficiency_{title.replace(' ', '_')}_data_subtract_bkg.pdf"
        else:
            title = f"efficiency_{title.replace(' ', '_')}_data.pdf"
        c.SaveAs(f"{path_hash}/{title}")
        c.Write(title, ROOT.TObject.kOverwrite)
        c.Close()
    root_file.Close()

Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure al

## Calc correction factor for $K_{S}\to\pi^+\pi^-$

In [15]:
corr_factor = {}

status = False
if delta_t_divided_data and delta_t_divided_signal:
    for hash in delta_t_divided_data:
        corr_factor[hash] = []
        for hist_data, hist_tag in zip(delta_t_divided_data[hash], delta_t_divided_signal[hash]):
            hist_temp = hist_data.Clone()
            hist_temp.Divide(hist_tag)
            corr_factor[hash].append(hist_temp)

    for hash in corr_factor:
        path_hash = f"{img_dir_path}/{hash}/efficiency_purity_plots"
        os.makedirs(path_hash, exist_ok=True)
        root_file_name = f"{path_hash}/eff_pur_corr_factor.root"
        root_file = ROOT.TFile(root_file_name, "UPDATE")
        root_file.mkdir("corr_factor")
        root_file.cd("corr_factor")
        for hist, title in zip(corr_factor[hash], delta_t_vars):
            c = ROOT.TCanvas("")
            hist.SetMarkerStyle(ROOT.kFullCircle)
            hist.SetLineColor(ROOT.kBlack)
            hist.GetYaxis().SetRangeUser(0, 1.1)
            hist.GetYaxis().SetTitle("Correction Factor")
            hist.Draw("PE1")
            if mode_eff == "subtract_bkg":
                hist.Write(f"corr_factor_{title.replace(' ', '_')}_subtract_bkg", ROOT.TObject.kOverwrite)
            else:
                hist.Write(f"corr_factor_{title.replace(' ', '_')}", ROOT.TObject.kOverwrite)
            c.Draw()
            if mode_eff == "subtract_bkg":
              title = f"corr_factor_{title.replace(' ', '_')}_subtract_bkg.pdf"
            else:
              title = f"corr_factor_{title.replace(' ', '_')}.pdf"
            c.SaveAs(f"{path_hash}/{title}")
            c.Write(title, ROOT.TObject.kOverwrite)
            c.Close()
        root_file.Close()

Error in <TFile::mkdir>: An object with name corr_factor exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/nocuts/efficiency_purity_plots/corr_factor_Delta_t_fit_subtract_bkg.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/nocuts/efficiency_purity_plots/corr_factor_Delta_t_rec_subtract_bkg.pdf has been created
Error in <TFile::mkdir>: An object with name corr_factor exists already
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/21f528cb/efficiency_purity_plots/corr_factor_Delta_t_fit_subtract_bkg.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/21f528cb/efficiency_purity_plots/corr_factor_Delta_t_rec_subtract_bkg.pdf has b

## Calculating events, efficiencies and purities

In [17]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(hist_file_path)

#################### OBJECT TO FIND ################################
event_counts = {}
cuts_list = {}
efficiencies = {}
purities = {}

#################### EXTRACT EVENT COUNTS #######################
for hash in hash_available:
    obj = hist_file.Get(f"{hash}/ChannelIntegrals")
    obj_cut_list = hist_file.Get(f"{hash}/CutList")

    if obj:
      chann_integrals_str = obj.GetTitle()
      event_counts[hash] = json.loads(chann_integrals_str)

    if obj_cut_list:
      raw_title = obj_cut_list.GetTitle()
      cleaned_str = raw_title.strip("[] ").replace('"', '')
      cuts_list[hash] = [c.strip() for c in cleaned_str.split(',') if c.strip()]

hist_file.Close()
#################################################################

#################### CALCULATE EFFICIENCIES AND PURITIES #######################
for hash in hash_available:
  if hash not in event_counts:
     continue

  efficiencies[hash] = {}
  purities[hash] = {}

  mc_sum = int(event_counts[hash].get("MC sum", 0))

  for flag in mc_flags.values():
      total_events = int(event_counts.get(nocut_hash, {}).get(flag, 0))
      passed_events = int(event_counts[hash].get(flag, 0))
      efficiencies[hash][flag] = passed_events / total_events if total_events > 0 else 0
      purities[hash][flag] = passed_events / mc_sum if mc_sum > 0 else 0
###############################################################################

################## PRINT RESULTS ##################
for hash in hash_available:
    if hash not in efficiencies:
        continue
    bkg = 0
    print(f"Results for cut list: {cuts_list.get(hash, [])}")
    for flag in mc_flags.values():
        event_count = int(event_counts.get(hash, {}).get(flag, 0))
        mc_sum = int(event_counts.get(hash, {}).get("MC sum", 0))
        eff = efficiencies[hash].get(flag, 0)
        pur = purities[hash].get(flag, 0)

        if flag != signal_channel and flag != "MC sum" and flag != "Data":
            bkg += event_count
        print(f"  Flag: {flag}, Event Count: {event_count / 10**6:.2f} x 10^6, Efficiency: {eff * 100:.2f}%, Purity: {pur * 100:.2f}%, (MC - Data) / Data: {100*(mc_sum - event_count) / event_count if event_count > 0 else 0:.2f}%")
    print(f"  Background (sum of non-signal, non-MC, non-Data flags): {bkg / 10**6:.2f} x 10^6")
    print()
    




Results for cut list: []
  Flag: Data, Event Count: 300.91 x 10^6, Efficiency: 152.94%, Purity: 99.86%, (MC - Data) / Data: 0.14%
  Flag: Signal, Event Count: 0.04 x 10^6, Efficiency: 126.21%, Purity: 0.01%, (MC - Data) / Data: 688434.23%
  Flag: Regeneration, Event Count: 0.62 x 10^6, Efficiency: 122.10%, Purity: 0.21%, (MC - Data) / Data: 48593.14%
  Flag: Omega, Event Count: 0.68 x 10^6, Efficiency: 40540.89%, Purity: 0.22%, (MC - Data) / Data: 44461.85%
  Flag: 3pi0, Event Count: 8.81 x 10^6, Efficiency: 118.19%, Purity: 2.92%, (MC - Data) / Data: 3319.79%
  Flag: Semileptonic, Event Count: 2.41 x 10^6, Efficiency: 98623.98%, Purity: 0.80%, (MC - Data) / Data: 12381.26%
  Flag: Other, Event Count: 288.27 x 10^6, Efficiency: 140.19%, Purity: 95.66%, (MC - Data) / Data: 4.53%
  Flag: pi+pi-pi+pi-, Event Count: 0.50 x 10^6, Efficiency: 116.30%, Purity: 0.17%, (MC - Data) / Data: 60022.87%
  Flag: MC sum, Event Count: 301.34 x 10^6, Efficiency: 140.77%, Purity: 100.00%, (MC - Data) / D

### Check if efficiencies / corr factors based on mc truth and tag are similar

In [35]:
import ROOT

cut_hash= "f63b399e"

metric = "corr_factor"
var = "Delta_t_rec"
ev_type = "MC"
histos = []
hist_div = None
hist_color = [ROOT.kRed, ROOT.kBlue]
leg_titles = ["Based on tag", "Based on MC truth"]
y_ranges = []
leg_position = ()
hist_title = ""

func_lin = ROOT.TF1("func_lin", "pol0", -300, 0)

if metric == "efficiency":
  dir_name = f"{metric}_{ev_type}"
  hist_name = f"{metric}_{var}_{ev_type}"
  y_ranges.append((0.0, 1.2))
  y_ranges.append((0.0, 1.2))
  leg_position = (0.54,0.70,0.85,0.9)
  hist_title = ev_type
else:
  dir_name = f"{metric}"
  hist_name = f"{metric}_{var}"
  y_ranges.append((0.0, 1.2))
  y_ranges.append((0.95, 1.05))
  leg_position = (0.15,0.02,0.46,0.22)

eff_histograms = f"results/{analysis_dir}/{analysis_hash}/img/{cut_hash}/efficiency_purity_plots/eff_pur_corr_factor.root"

root_file = ROOT.TFile.Open(eff_histograms)

eff_dir = root_file.Get(dir_name)
histos.append(eff_dir.Get(hist_name))
histos.append(eff_dir.Get(f"{hist_name}_subtract_bkg"))

hist_div = histos[0].Clone("hist_div")
hist_div.Divide(histos[1])
hist_div.SetLineColor(ROOT.kBlack)
hist_div.SetMarkerColor(ROOT.kBlack)
hist_div.GetYaxis().SetRangeUser(*y_ranges[1])

leg = ROOT.TLegend(*leg_position)

can = ROOT.TCanvas("c1","c1",700, 1000)

# TPad(name, title, xlow, ylow, xup, yup) - współrzędne od 0 do 1
# Górny pad (np. od 30% do 100% wysokości kanwy)
pad1 = ROOT.TPad("pad1", "pad1", 0.0, 0.3, 1.0, 1.0)
pad1.SetBottomMargin(0.02)  # Minimalny margines od dołu, żeby ukryć etykiety X
pad1.SetTopMargin(0.1)
pad1.Draw()

# Dolny pad (np. od 0% do 30% wysokości kanwy)
pad2 = ROOT.TPad("pad2", "pad2", 0.0, 0.0, 1.0, 0.3)
pad2.SetTopMargin(0.05)     # Minimalny margines od góry
pad2.SetBottomMargin(0.35)   # Większy margines na etykiety osi X dolnego wykresów
pad2.SetGrid()
pad2.Draw()

for i, hist in enumerate(histos):
    hist.SetLineColor(hist_color[i])
    hist.SetMarkerColor(hist_color[i])
    hist.GetYaxis().SetRangeUser(*y_ranges[0])
    hist.SetTitle(hist_title)
    leg.AddEntry(hist, leg_titles[i], "lp")

pad1.cd()
histos[0].Draw()
histos[1].Draw("SAME")

leg.Draw()

pad2.cd()

hist_div.Fit(func_lin, "QSR")
hist_div.SetTitle("") 
hist_div.GetYaxis().SetTitle("Ratio")
hist_div.GetYaxis().SetNdivisions(505) # Mniej kresek na osi Y, żeby się nie zlewały

# Kluczowe: skalowanie czcionek dla dolnego (mniejszego) padu
# Ponieważ pad ma 30% wysokości, czcionki muszą być powiększone współczynnikiem ~1/0.3
scale_factor = 1.0 / 0.45

hist_div.GetYaxis().SetLabelSize(0.04 * scale_factor)
hist_div.GetYaxis().SetTitleSize(0.05 * scale_factor)
hist_div.GetYaxis().SetTitleOffset(0.6) # Odsuwamy napis "Ratio", żeby nie nachodził na cyfry

hist_div.GetXaxis().SetLabelSize(0.04 * scale_factor)
hist_div.GetXaxis().SetTitleSize(0.05 * scale_factor)
hist_div.GetXaxis().SetTitleOffset(1.0)

# Upewnienie się, że zakresy osi X są identyczne na obu padach
hist_div.GetXaxis().SetRange(histos[0].GetXaxis().GetFirst(), histos[0].GetXaxis().GetLast())

hist_div.Draw("E1")
func_lin.Draw("SAME")

text = ROOT.TPaveText(0.15, 0.8, 0.4, 0.95, "NDC")
text.SetTextSize(0.02 * scale_factor)
text.AddText(f"Fit: {func_lin.GetParameter(0):.3f} #pm {func_lin.GetParError(0):.3f}")
text.AddText(f"Chi2/NDF: {func_lin.GetChisquare():.3f}/{func_lin.GetNDF()}")
text.Draw("SAME")

can.Draw()

name = f"{metric}_{var}_tag_vs_mc_truth_{ev_type}.pdf" if ev_type else f"{metric}_{var}_tag_vs_mc_truth.pdf"
can.SaveAs(f"results/{analysis_dir}/{analysis_hash}/img/{cut_hash}/efficiency_purity_plots/{name}")

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: pdf file results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/f63b399e/efficiency_purity_plots/corr_factor_Delta_t_rec_tag_vs_mc_truth_MC.pdf has been created


In [74]:
import ROOT

cut_hash= "f63b399e"

var = "Delta_t_rec"
ev_type = ["data", "MC"]
histos_eff = []
hist_corr = None
hist_color = [ROOT.kRed, ROOT.kBlue]
leg_titles = ["Data", "MC"]
y_ranges = []
leg_position = (0.66, 0.82, 0.85, 0.9)
hist_title = ""

y_ranges = [(0, 1.1), (0.5, 1.3)]

eff_histograms = f"results/{analysis_dir}/{analysis_hash}/img/{cut_hash}/efficiency_purity_plots/eff_pur_corr_factor.root"

root_file = ROOT.TFile.Open(eff_histograms)

for et in ev_type:
    eff_dir = root_file.Get(f"efficiency_{et}")
    histos_eff.append(eff_dir.Get(f"efficiency_{var}_{et}"))

corr_dir = root_file.Get("corr_factor")
hist_corr = corr_dir.Get(f"corr_factor_{var}")

leg = ROOT.TLegend(*leg_position)

can = ROOT.TCanvas("c1","c1",700, 1000)

# TPad(name, title, xlow, ylow, xup, yup) - współrzędne od 0 do 1
# Górny pad (np. od 30% do 100% wysokości kanwy)
pad1 = ROOT.TPad("pad1", "pad1", 0.0, 0.3, 1.0, 1.0)
pad1.SetBottomMargin(0.02)  # Minimalny margines od dołu, żeby ukryć etykiety X
pad1.SetTopMargin(0.1)
pad1.Draw()

# Dolny pad (np. od 0% do 30% wysokości kanwy)
pad2 = ROOT.TPad("pad2", "pad2", 0.0, 0.0, 1.0, 0.3)
pad2.SetTopMargin(0.05)     # Minimalny margines od góry
pad2.SetBottomMargin(0.35)   # Większy margines na etykiety osi X dolnego wykresów
pad2.SetGrid()
pad2.Draw()

for i, hist in enumerate(histos_eff):
    hist.SetLineColor(hist_color[i])
    hist.SetMarkerColor(hist_color[i])
    hist.GetYaxis().SetRangeUser(*y_ranges[0])
    hist.SetTitle(hist_title)
    leg.AddEntry(hist, leg_titles[i], "lp")

pad1.cd()
histos_eff[0].Draw()
histos_eff[1].Draw("SAME")

leg.Draw()

pad2.cd()

hist_corr.GetYaxis().SetTitle("Data / MC")
hist_corr.SetLineColor(ROOT.kBlack)
hist_corr.SetMarkerColor(ROOT.kBlack)
hist_corr.GetYaxis().SetRangeUser(*y_ranges[1])

hist_corr.SetTitle("") 
hist_corr.GetYaxis().SetNdivisions(505) # Mniej kresek na osi Y, żeby się nie zlewały

# Kluczowe: skalowanie czcionek dla dolnego (mniejszego) padu
# Ponieważ pad ma 30% wysokości, czcionki muszą być powiększone współczynnikiem ~1/0.3
scale_factor = 1.0 / 0.45

hist_corr.GetYaxis().SetLabelSize(0.04 * scale_factor)
hist_corr.GetYaxis().SetTitleSize(0.05 * scale_factor)
hist_corr.GetYaxis().SetTitleOffset(0.6) # Odsuwamy napis "Ratio", żeby nie nachodził na cyfry

hist_corr.GetXaxis().SetLabelSize(0.04 * scale_factor)
hist_corr.GetXaxis().SetTitleSize(0.05 * scale_factor)
hist_corr.GetXaxis().SetTitleOffset(1.0)

# Upewnienie się, że zakresy osi X są identyczne na obu padach
hist_corr.GetXaxis().SetRange(histos_eff[0].GetXaxis().GetFirst(), histos_eff[0].GetXaxis().GetLast())

hist_corr.Draw("E1")

can.Draw()

name = f"{var}_corr_factor_final.pdf"
can.SaveAs(f"results/{analysis_dir}/{analysis_hash}/img/{cut_hash}/efficiency_purity_plots/{name}")

Info in <TCanvas::Print>: pdf file results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/f63b399e/efficiency_purity_plots/Delta_t_rec_corr_factor_final.pdf has been created


### Comparison of variables between control sample pairs

* Comparison of correction factors for both control samples

In [10]:
import math
import ROOT

ROOT.gROOT.SetBatch(True)

output_dir = "comparison"
hist_type = ""
metric = "corr_factor"
suffix = ""
analysis_pairs = [("ks_pm_control_sample", "kl_pm_control_sample")]
analysis_hash_pairs = [("c97f42f6a601db1c64b593e259e04dec", "7b6c187f3e4b0390d51437281fd4292b")]
cut_hash = [("03e630a8", "f63b399e")]
var_pairs = [("Delta_t_tri_fit", "Delta_t_fit")]
zoom_mode = "full"
channels = [("3pi0", "pi+pi-pi+pi-")]

enable_spline = True      # główny przełącznik włączający/wyłączający TSpline3
enable_spline_h0 = True   # opcjonalnie osobno dla h0
enable_spline_h1 = True   # opcjonalnie osobno dla h1

rebin_factor = 6        # ile sąsiednich binów histogramu łączyć w jeden
spline_group = 1          # po ile punktów z TGraphErrors uśredniać (1/err^2) przed TSpline3 (1 = bez dodatkowego łączenia)
split_at_zero = False     # osobny TSpline3 dla x < 0 i x >= 0 (chroni okolice 0 dla Im)

ind = 0
tested_pair = analysis_pairs[ind]
analysis_hash = analysis_hash_pairs[ind]
cut_pair = cut_hash[ind]
var = var_pairs[ind]
channel = channels[ind]

output_dir = f"comparison/{tested_pair[0]}_{tested_pair[1]}/{cut_pair[0]}_{cut_pair[1]}/{var[0]}_{var[1]}"
import os
os.makedirs(output_dir, exist_ok=True)

hist_files = []
histograms = []

for i in range(2):
    f = ROOT.TFile.Open(
        f"results/{tested_pair[i]}/{analysis_hash[i]}/img/{cut_pair[i]}/efficiency_purity_plots/eff_pur_corr_factor.root",
        "READ",
    )
    hist_files.append(f)

    hist_name = f"{metric}/{metric}_{var[i]}_{suffix}" if suffix else f"{metric}/{metric}_{var[i]}"
    h = f.Get(hist_name)
    h.SetDirectory(0)
    histograms.append(h)


def rebin_ratio_hist(h, ngroup, new_name):
    """Łączy ngroup binów dla histogramu stosunku/współczynnika (średnia ważona 1/err^2)."""
    if ngroup <= 1:
        h_out = h.Clone(new_name)
        h_out.SetDirectory(0)
        return h_out

    n_bins = h.GetNbinsX()
    n_new = n_bins // ngroup
    x_min = h.GetXaxis().GetBinLowEdge(1)
    x_max = h.GetXaxis().GetBinUpEdge(n_new * ngroup)

    h_out = ROOT.TH1D(new_name, h.GetTitle(), n_new, x_min, x_max)
    h_out.SetDirectory(0)

    for i_new in range(1, n_new + 1):
        b_start = (i_new - 1) * ngroup + 1
        b_end = i_new * ngroup

        sum_w = 0.0
        sum_wy = 0.0

        for b in range(b_start, b_end + 1):
            y = h.GetBinContent(b)
            ey = h.GetBinError(b)
            if ey > 0 and math.isfinite(y) and math.isfinite(ey):
                w = 1.0 / (ey * ey)
                sum_w += w
                sum_wy += w * y

        if sum_w > 0:
            h_out.SetBinContent(i_new, sum_wy / sum_w)
            h_out.SetBinError(i_new, 1.0 / math.sqrt(sum_w))
        else:
            h_out.SetBinContent(i_new, 0.0)
            h_out.SetBinError(i_new, 0.0)

    return h_out


hist_res = [
    rebin_ratio_hist(histograms[0], rebin_factor, "h0_rebin"),
    rebin_ratio_hist(histograms[1], rebin_factor, "h1_rebin"),
]


def make_clean_graph(h, max_err=0.25, max_rel_err=0.35):
    g = ROOT.TGraphErrors()
    idx = 0
    for b in range(1, h.GetNbinsX() + 1):
        x = h.GetBinCenter(b)
        ex = h.GetBinWidth(b) / 2.0
        y = h.GetBinContent(b)
        ey = h.GetBinError(b)

        if ey <= 0 or y == 0:
            continue
        if ey > max_err or (ey / abs(y)) > max_rel_err:
            continue

        g.SetPoint(idx, x, y)
        g.SetPointError(idx, ex, ey)
        idx += 1
    return g


def group_points_for_spline(pts, group_size):
    """Opcjonalnie łączy po group_size punktów średnią ważoną 1/err^2 (wymaga ściśle rosnącego X dla TSpline3)."""
    if group_size <= 1 or len(pts) < group_size:
        return pts

    grouped = []
    for start in range(0, len(pts) - group_size + 1, group_size):
        chunk = pts[start : start + group_size]
        x_total = 0.0
        for p in chunk:
            x_total += p[0]
        x_mean = x_total / len(chunk)

        sum_w = 0.0
        sum_wy = 0.0
        for _, y, e in chunk:
            w = 1.0 / (e * e)
            sum_w += w
            sum_wy += w * y
        if sum_w > 0:
            y_mean = sum_wy / sum_w
            e_mean = 1.0 / math.sqrt(sum_w)
            if not grouped or x_mean > grouped[-1][0]:
                grouped.append((x_mean, y_mean, e_mean))
    return grouped


def build_tspline3_list(g, name_prefix, split_zero=True, group_size=1, enabled=True):
    """Buduje listę obiektów ROOT.TSpline3 (osobno dla x < 0 i x >= 0 lub całościowo)."""
    if not enabled:
        return []

    pts = []
    for i in range(g.GetN()):
        xf = float(g.GetPointX(i))
        yf = float(g.GetPointY(i))
        eyf = float(g.GetErrorY(i))
        if eyf > 0 and math.isfinite(xf) and math.isfinite(yf) and math.isfinite(eyf):
            pts.append((xf, yf, eyf))

    pts.sort(key=lambda p: p[0])

    # Usuwanie ewentualnych duplikatów X (TSpline3 wymaga ściśle rosnącego X)
    unique_pts = []
    for p in pts:
        if not unique_pts or p[0] > unique_pts[-1][0]:
            unique_pts.append(p)

    segments = []
    if split_zero:
        neg = group_points_for_spline([p for p in unique_pts if p[0] < 0.0], group_size)
        pos = group_points_for_spline([p for p in unique_pts if p[0] >= 0.0], group_size)
        if len(neg) >= 3:
            segments.append(neg)
        if len(pos) >= 3:
            segments.append(pos)
    else:
        all_pts = group_points_for_spline(unique_pts, group_size)
        if len(all_pts) >= 3:
            segments.append(all_pts)

    splines = []
    for seg_idx, seg in enumerate(segments):
        g_knots = ROOT.TGraph(len(seg))
        for j, (px, py, _) in enumerate(seg):
            g_knots.SetPoint(j, px, py)

        sp = ROOT.TSpline3(f"{name_prefix}_spline3_{seg_idx}", g_knots)
        ROOT.SetOwnership(sp, True)
        splines.append((sp, g_knots))  # trzymamy g_knots przy życiu razem ze splajnem

    return splines


g0 = make_clean_graph(hist_res[0], max_err=0.02, max_rel_err=0.35)
g1 = make_clean_graph(hist_res[1], max_err=0.09, max_rel_err=0.35)

spl0_items = build_tspline3_list(
    g0, "h0", split_zero=split_at_zero, group_size=spline_group, enabled=(enable_spline and enable_spline_h0)
)
spl1_items = build_tspline3_list(
    g1, "h1", split_zero=split_at_zero, group_size=spline_group, enabled=(enable_spline and enable_spline_h1)
)

ROOT.TGaxis.SetMaxDigits(5)
c = ROOT.TCanvas("c", "c", 800, 800)

g0.SetTitle(f"; #Deltat [#tau_{{S}}]; Correction Factor")
g0.SetLineColor(ROOT.kRed)
g0.SetMarkerColor(ROOT.kRed)
g0.SetMarkerStyle(20)
g0.GetYaxis().SetRangeUser(0.3, 1.5)
g0.GetXaxis().SetLimits(-320, 320)

g1.SetLineColor(ROOT.kBlue)
g1.SetMarkerColor(ROOT.kBlue)
g1.SetMarkerStyle(20)

g0.Draw("AP")
g1.Draw("P SAME")

for sp, _ in spl0_items:
    sp.SetLineColor(ROOT.kRed + 2)
    sp.SetLineWidth(3)
    sp.Draw("C SAME")

for sp, _ in spl1_items:
    sp.SetLineColor(ROOT.kBlue + 2)
    sp.SetLineWidth(3)
    sp.Draw("C SAME")

leg = ROOT.TLegend(0.15, 0.78, 0.4, 0.9)
leg.AddEntry(g0, "K_{S}#rightarrow#pi^{+}#pi^{-}", "lp")
leg.AddEntry(g1, "K_{L}#rightarrow#pi^{+}#pi^{-}", "lp")
if spl0_items:
    leg.AddEntry(spl0_items[0][0], "TSpline3 h0", "l")
if spl1_items:
    leg.AddEntry(spl1_items[0][0], "TSpline3 h1", "l")
leg.Draw()

c.Draw()
c.SaveAs(f"{output_dir}/correction_factor_plot_rebin_{rebin_factor}_spline_{enable_spline}.pdf")

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c
Info in <TCanvas::Print>: pdf file comparison/ks_pm_control_sample_kl_pm_control_sample/03e630a8_f63b399e/Delta_t_tri_fit_Delta_t_fit/correction_factor_plot_rebin_6_spline_True.pdf has been created


In [34]:
import math
import ROOT

ROOT.gROOT.SetBatch(True)

hist_type = ""
metric = "corr_factor"
suffix = ""
analysis_pairs = [("ks_pm_control_sample", "kl_pm_control_sample")]
analysis_hash_pairs = [("c97f42f6a601db1c64b593e259e04dec", "7b6c187f3e4b0390d51437281fd4292b")]
cut_hash = [("03e630a8", "f63b399e")]
var_pairs = [("Delta_t_tri_fit", "Delta_t_fit")]
zoom_mode = "full"
channels = [("3pi0", "pi+pi-pi+pi-")]

rebin_factor = 1       # ile sąsiednich binów histogramu łączyć w jeden
split_at_zero = True   # osobny wielomian dla x < 0 i x >= 0 (chroni okolice 0 dla Im)

# Osobne stopnie wielomianów dla h0 i h1 (przed 0 i po 0)
h0_deg_neg = 4
h0_deg_pos = 4
h1_deg_neg = 4
h1_deg_pos = 9

poly_deg_per_hist = [
    (h0_deg_neg, h0_deg_pos),
    (h1_deg_neg, h1_deg_pos),
]

ind = 0
tested_pair = analysis_pairs[ind]
analysis_hash = analysis_hash_pairs[ind]
cut_pair = cut_hash[ind]
var = var_pairs[ind]
channel = channels[ind]

hist_files = []
histograms = []

for i in range(2):
    f = ROOT.TFile.Open(
        f"results/{tested_pair[i]}/{analysis_hash[i]}/img/{cut_pair[i]}/efficiency_purity_plots/eff_pur_corr_factor.root",
        "READ",
    )
    hist_files.append(f)

    hist_name = f"{metric}/{metric}_{var[i]}_{suffix}" if suffix else f"{metric}/{metric}_{var[i]}"
    h = f.Get(hist_name)
    h.SetDirectory(0)
    histograms.append(h)


def rebin_ratio_hist(h, ngroup, new_name):
    """Łączy ngroup binów dla histogramu stosunku/współczynnika (średnia ważona 1/err^2)."""
    if ngroup <= 1:
        h_out = h.Clone(new_name)
        h_out.SetDirectory(0)
        return h_out

    n_bins = h.GetNbinsX()
    n_new = n_bins // ngroup
    x_min = h.GetXaxis().GetBinLowEdge(1)
    x_max = h.GetXaxis().GetBinUpEdge(n_new * ngroup)

    h_out = ROOT.TH1D(new_name, h.GetTitle(), n_new, x_min, x_max)
    h_out.SetDirectory(0)

    for i_new in range(1, n_new + 1):
        b_start = (i_new - 1) * ngroup + 1
        b_end = i_new * ngroup

        sum_w = 0.0
        sum_wy = 0.0

        for b in range(b_start, b_end + 1):
            y = h.GetBinContent(b)
            ey = h.GetBinError(b)
            if ey > 0 and math.isfinite(y) and math.isfinite(ey):
                w = 1.0 / (ey * ey)
                sum_w += w
                sum_wy += w * y

        if sum_w > 0:
            h_out.SetBinContent(i_new, sum_wy / sum_w)
            h_out.SetBinError(i_new, 1.0 / math.sqrt(sum_w))
        else:
            h_out.SetBinContent(i_new, 0.0)
            h_out.SetBinError(i_new, 0.0)

    return h_out


hist_res = [
    rebin_ratio_hist(histograms[0], rebin_factor, "h0_rebin"),
    rebin_ratio_hist(histograms[1], rebin_factor, "h1_rebin"),
]


def make_clean_graph(h, max_err=0.25, max_rel_err=0.35):
    g = ROOT.TGraphErrors()
    idx = 0
    for b in range(1, h.GetNbinsX() + 1):
        x = h.GetBinCenter(b)
        ex = h.GetBinWidth(b) / 2.0
        y = h.GetBinContent(b)
        ey = h.GetBinError(b)

        if ey <= 0 or y == 0:
            continue
        if ey > max_err or (ey / abs(y)) > max_rel_err:
            continue

        g.SetPoint(idx, x, y)
        g.SetPointError(idx, ex, ey)
        idx += 1
    return g


def build_poly_fits(g, name_prefix, deg_neg=4, deg_pos=4, split_zero=True):
    """Dopasowuje wielomian stopnia `deg_neg` (x < 0) oraz `deg_pos` (x >= 0) z wagami 1/ey^2."""
    pts = []
    for i in range(g.GetN()):
        xf = float(g.GetPointX(i))
        yf = float(g.GetPointY(i))
        eyf = float(g.GetErrorY(i))
        if eyf > 0 and math.isfinite(xf) and math.isfinite(yf) and math.isfinite(eyf):
            pts.append((xf, yf, eyf))

    pts.sort(key=lambda p: p[0])

    segments = []
    if split_zero:
        neg = [p for p in pts if p[0] < 0.0]
        pos = [p for p in pts if p[0] >= 0.0]
        if len(neg) > deg_neg + 1:
            segments.append(("neg", neg, deg_neg))
        if len(pos) > deg_pos + 1:
            segments.append(("pos", pos, deg_pos))
    else:
        deg_all = deg_pos if len(pts) > 0 and pts[-1][0] >= 0.0 else deg_neg
        if len(pts) > deg_all + 1:
            segments.append(("all", pts, deg_all))

    fits = []
    for seg_idx, (side_label, seg, deg) in enumerate(segments):
        g_sub = ROOT.TGraphErrors(len(seg))
        for j, (px, py, pe) in enumerate(seg):
            g_sub.SetPoint(j, px, py)
            g_sub.SetPointError(j, 0.0, pe)

        x_min = seg[0][0]
        x_max = seg[-1][0]
        tf = ROOT.TF1(f"{name_prefix}_poly{deg}_{side_label}_{seg_idx}", f"pol{deg}", x_min, x_max)
        tf.SetNpx(500)

        g_sub.Fit(tf, "QRS")
        ROOT.SetOwnership(tf, True)
        fits.append((tf, g_sub, side_label, deg))

    return fits


g0 = make_clean_graph(hist_res[0], max_err=0.02, max_rel_err=0.35)
g1 = make_clean_graph(hist_res[1], max_err=0.08, max_rel_err=0.35)

fit0_items = build_poly_fits(
    g0, "h0", deg_neg=poly_deg_per_hist[0][0], deg_pos=poly_deg_per_hist[0][1], split_zero=split_at_zero
)
fit1_items = build_poly_fits(
    g1, "h1", deg_neg=poly_deg_per_hist[1][0], deg_pos=poly_deg_per_hist[1][1], split_zero=split_at_zero
)

c = ROOT.TCanvas("c", "c", 800, 600)

g0.SetTitle(f"{metric}; {var[0]} [#tau_{{S}}]; Correction Factor")
g0.SetLineColor(ROOT.kRed)
g0.SetMarkerColor(ROOT.kRed)
g0.SetMarkerStyle(20)
g0.GetYaxis().SetRangeUser(0, 2)
g0.GetXaxis().SetLimits(-320, 320)

g1.SetLineColor(ROOT.kBlue)
g1.SetMarkerColor(ROOT.kBlue)
g1.SetMarkerStyle(20)

g0.Draw("AP")
g1.Draw("P SAME")

for tf, _, _, _ in fit0_items:
    tf.SetLineColor(ROOT.kRed + 2)
    tf.SetLineWidth(3)
    tf.Draw("SAME")

for tf, _, _, _ in fit1_items:
    tf.SetLineColor(ROOT.kBlue + 2)
    tf.SetLineWidth(3)
    tf.Draw("SAME")

leg = ROOT.TLegend(0.52, 0.70, 0.88, 0.88)
leg.AddEntry(g0, f"{tested_pair[0]} ({var[0]})", "lp")
leg.AddEntry(g1, f"{tested_pair[1]} ({var[1]})", "lp")
if fit0_items:
    desc0 = ", ".join(f"{side}:pol{d}" for _, _, side, d in fit0_items)
    leg.AddEntry(fit0_items[0][0], f"h0 ({desc0})", "l")
if fit1_items:
    desc1 = ", ".join(f"{side}:pol{d}" for _, _, side, d in fit1_items)
    leg.AddEntry(fit1_items[0][0], f"h1 ({desc1})", "l")
leg.Draw()

c.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c


In [96]:
import os
import ROOT

ROOT.gROOT.SetBatch(True)

hist_type = ""
analysis_pairs = [("ks_pm_control_sample", "kl_pm_control_sample")]
analysis_hash_pairs = [("c97f42f6a601db1c64b593e259e04dec", "7b6c187f3e4b0390d51437281fd4292b")]
cut_hash = [("03e630a8", "f63b399e")]
var_pairs = [("R_ch", "R_KL")]
zoom_mode = "full"
channels = [("3pi0", "pi+pi-pi+pi-")]

rebin_factor = 1    # 1 = bez rebinningu, >1 = łączenie sąsiednich binów w histogramach R_ch / R_KL
enable_fits = True  # główny przełącznik dopasowań KS i KL
enable_fit_ks = True
enable_fit_kl = True

run_fit_ks = enable_fits and enable_fit_ks
run_fit_kl = enable_fits and enable_fit_kl

ind = 0
tested_pair = analysis_pairs[ind]
analysis_hash = analysis_hash_pairs[ind]
cut_pair = cut_hash[ind]
var = var_pairs[ind]
channel = channels[ind]

output_dir = f"comparison/{tested_pair[0]}_{tested_pair[1]}/{cut_pair[0]}_{cut_pair[1]}/{var[0]}_{var[1]}"
os.makedirs(output_dir, exist_ok=True)

hist_files = []
histograms = []

for i in range(2):
    f = ROOT.TFile.Open(
        f"results/{tested_pair[i]}/{analysis_hash[i]}/img/{cut_pair[i]}/histograms.root",
        "READ",
    )
    hist_files.append(f)
    h = f.Get(f"{var[i]}/{var[i]}_{channel[i]}_{zoom_mode}")
    h.SetDirectory(0)
    histograms.append(h)

hist_res = [histograms[0].Clone("h0_copy"), histograms[1].Clone("h1_copy")]
for h in hist_res:
    h.SetDirectory(0)
    if rebin_factor > 1:
        h.Rebin(rebin_factor)

# 1. Dopasowanie KS i KL (zakres 12..150 cm)
func = None
func1 = None
fit_res_0 = None
fit_res_1 = None

if run_fit_ks:
    func = ROOT.TF1(
        "decay",
        "[0] * ([1] * exp(-x / [2]) + (1.0 - [1]) * exp(-x / [3])) + [4]",
        12.0,
        150.0,
    )
    func.SetParameters(hist_res[0].GetMaximum(), 0.9, 0.55, 10.0, 10.0)
    func.SetParLimits(1, 0.0, 1.0)
    func.SetParLimits(2, 0.05, 10.0)
    func.SetParLimits(3, 1.0, 100.0)
    func.SetParLimits(4, 0.0, 1e4)
    fit_res_0 = hist_res[0].Fit(func, "RLS")
    if func.GetNDF() > 0:
        print(f"KS fit: chi2/ndf = {func.GetChisquare() / func.GetNDF():.2f}")

if run_fit_kl:
    func1 = ROOT.TF1(
        "decay1",
        "[0] * ([1] * exp(-x / [2]) + (1.0 - [1]) * exp(-x / [3])) + [4]",
        12.0,
        150.0,
    )
    func1.SetParameters(hist_res[1].GetMaximum(), 0.5, 2.0, 200.0, 0.0)
    func1.SetParLimits(1, 0.0, 1.0)
    func1.SetParLimits(2, 0.1, 20.0)
    func1.SetParLimits(3, 20.0, 1000.0)
    func1.SetParLimits(4, -1e3, 1e4)
    fit_res_1 = hist_res[1].Fit(func1, "RLS")
    if func1.GetNDF() > 0:
        print(f"KL fit: chi2/ndf = {func1.GetChisquare() / func1.GetNDF():.2f}")


def make_residuals(hist, tf1, xmin, xmax, name_prefix):
    """Zwraca histogramy (reszta_bezwzgledna, pull) w zadanym zakresie [xmin, xmax]."""
    h_diff = hist.Clone(f"{name_prefix}_diff")
    h_pull = hist.Clone(f"{name_prefix}_pull")
    h_diff.SetDirectory(0)
    h_pull.SetDirectory(0)
    h_diff.Reset()
    h_pull.Reset()

    for b in range(1, hist.GetNbinsX() + 1):
        xc = hist.GetBinCenter(b)
        if xc < xmin or xc > xmax:
            continue
        val = hist.GetBinContent(b)
        err = hist.GetBinError(b)
        pred = tf1.Eval(xc)
        diff = val - pred
        h_diff.SetBinContent(b, diff)
        h_diff.SetBinError(b, err)
        if err > 0:
            h_pull.SetBinContent(b, diff / err)
            h_pull.SetBinError(b, 1.0)

    return h_diff, h_pull


h0_diff = None
h0_pull = None
h1_diff = None
h1_pull = None

if run_fit_ks and func is not None:
    h0_diff, h0_pull = make_residuals(hist_res[0], func, 12.0, 150.0, "ks")
if run_fit_kl and func1 is not None:
    h1_diff, h1_pull = make_residuals(hist_res[1], func1, 12.0, 150.0, "kl")

has_any_pull = (h0_pull is not None) or (h1_pull is not None)

# Zamknięcie ewentualnych starszych płócien o tej samej nazwie
for cname in ("c_dists", "c"):
    old_c = ROOT.gROOT.FindObject(cname)
    if old_c:
        old_c.Close()

# Sprawdzenie czy w zakresie [0, 150] występują ujemne wartości (blokują skalę logarytmiczną Y)
min_val_in_range = None
for h in hist_res:
    for b in range(1, h.GetNbinsX() + 1):
        xc = h.GetBinCenter(b)
        if 0.0 <= xc <= 150.0:
            v = h.GetBinContent(b)
            if min_val_in_range is None or v < min_val_in_range:
                min_val_in_range = v

use_logy = (min_val_in_range is None) or (min_val_in_range >= 0.0)

# 1b. Rysowanie pierwotnych rozkładów (KS i KL) wraz z dopasowanymi funkcjami
c_dists = ROOT.TCanvas("c_dists", "c_dists", 700, 700)
if has_any_pull:
    pad_d1 = ROOT.TPad("pad_d1", "pad_d1", 0.0, 0.35, 1.0, 1.0)
    pad_d2 = ROOT.TPad("pad_d2", "pad_d2", 0.0, 0.0, 1.0, 0.35)
    pad_d1.SetBottomMargin(0.02)
    pad_d2.SetTopMargin(0.02)
    pad_d2.SetBottomMargin(0.28)
    pad_d1.Draw()
    pad_d2.Draw()
else:
    pad_d1 = ROOT.TPad("pad_d1", "pad_d1", 0.0, 0.0, 1.0, 1.0)
    pad_d2 = None
    pad_d1.Draw()

pad_d1.cd()
pad_d1.SetLogy(use_logy)

hist_res[0].SetLineColor(ROOT.kRed)
hist_res[0].SetMarkerColor(ROOT.kRed)
hist_res[0].SetMarkerStyle(20)
hist_res[0].SetMarkerSize(0.7)
hist_res[0].SetTitle("; R_{+-,IP} [cm]; Entries")
hist_res[0].GetXaxis().SetRangeUser(0.0, 150.0)
y_max_dist = max(hist_res[0].GetMaximum(), hist_res[1].GetMaximum()) * 3.0
hist_res[0].SetMinimum(1.0 if use_logy else min(0.0, (min_val_in_range or 0.0) * 1.1))
hist_res[0].SetMaximum(y_max_dist)

hist_res[1].SetLineColor(ROOT.kBlue)
hist_res[1].SetMarkerColor(ROOT.kBlue)
hist_res[1].SetMarkerStyle(21)
hist_res[1].SetMarkerSize(0.7)
hist_res[1].SetMinimum(1.0 if use_logy else min(0.0, (min_val_in_range or 0.0) * 1.1))

hist_res[0].Draw("PE")
hist_res[1].Draw("PE1 SAME")

if run_fit_ks and func is not None:
    func.SetLineColor(ROOT.kOrange + 7)
    func.SetLineWidth(2)
    func.SetNpx(1000)
    func.Draw("SAME")

if run_fit_kl and func1 is not None:
    func1.SetLineColor(ROOT.kCyan + 2)
    func1.SetLineWidth(2)
    func1.SetNpx(1000)
    func1.Draw("SAME")

leg_dists = ROOT.TLegend(0.6, 0.8, 0.85, 0.9)
leg_dists.AddEntry(hist_res[0], "K_{S}#rightarrow#pi^{+}#pi^{-}", "lp")
if run_fit_ks and func is not None:
    leg_dists.AddEntry(func, "Fit K_{S}", "l")
leg_dists.AddEntry(hist_res[1], "K_{L}#rightarrow#pi^{+}#pi^{-}", "lp")
if run_fit_kl and func1 is not None:
    leg_dists.AddEntry(func1, "Fit K_{L}", "l")
leg_dists.Draw()

line_pull_zero = None
if has_any_pull and pad_d2 is not None:
    pad_d2.cd()
    first_pull = True
    if h0_pull is not None:
        h0_pull.SetLineColor(ROOT.kOrange + 7)
        h0_pull.SetMarkerColor(ROOT.kOrange + 7)
        h0_pull.SetMarkerStyle(20)
        h0_pull.SetMarkerSize(0.6)
        h0_pull.GetXaxis().SetRangeUser(0.0, 150.0)
        h0_pull.GetYaxis().SetRangeUser(-5.0, 5.0)
        h0_pull.GetYaxis().SetTitle("Pull")
        h0_pull.GetYaxis().SetTitleSize(0.08)
        h0_pull.GetYaxis().SetLabelSize(0.07)
        h0_pull.GetXaxis().SetTitleSize(0.08)
        h0_pull.GetXaxis().SetLabelSize(0.07)
        h0_pull.Draw("E")
        first_pull = False

    if h1_pull is not None:
        h1_pull.SetLineColor(ROOT.kCyan + 2)
        h1_pull.SetMarkerColor(ROOT.kCyan + 2)
        h1_pull.SetMarkerStyle(21)
        h1_pull.SetMarkerSize(0.6)
        if first_pull:
            h1_pull.GetXaxis().SetRangeUser(0.0, 150.0)
            h1_pull.GetYaxis().SetRangeUser(-5.0, 5.0)
            h1_pull.GetYaxis().SetTitle("Pull")
            h1_pull.GetYaxis().SetTitleSize(0.08)
            h1_pull.GetYaxis().SetLabelSize(0.07)
            h1_pull.GetXaxis().SetTitleSize(0.08)
            h1_pull.GetXaxis().SetLabelSize(0.07)
            h1_pull.Draw("E")
        else:
            h1_pull.Draw("E SAME")

    line_pull_zero = ROOT.TLine(0.0, 0.0, 150.0, 0.0)
    line_pull_zero.SetLineStyle(2)
    line_pull_zero.Draw("SAME")

c_dists.SaveAs(f"{output_dir}/decay_distance_{rebin_factor}_{channel[0]}_{channel[1]}_{enable_fits}.pdf")

# 2. Ratio z histogramów i funkcji (po ewentualnym rebinningu w hist_res)
hist_sum = hist_res[0].Clone("hist_sum")
hist_sum.SetDirectory(0)
hist_sum.Add(hist_res[1])

hist_ratio_0 = hist_res[0].Clone("hist_ratio_0")
hist_ratio_0.SetDirectory(0)
hist_ratio_0.Divide(hist_sum)

hist_ratio_1 = hist_res[1].Clone("hist_ratio_1")
hist_ratio_1.SetDirectory(0)
hist_ratio_1.Divide(hist_sum)

has_full_ratio_fit = run_fit_ks and run_fit_kl and (func is not None) and (func1 is not None)
ratio_1 = None
ratio_2 = None
h_ratio0_res = None
h_ratio1_res = None

if has_full_ratio_fit:
    expr_ks = "([0] * ([1] * exp(-x / [2]) + (1.0 - [1]) * exp(-x / [3])) + [4])"
    expr_kl = "([5] * ([6] * exp(-x / [7]) + (1.0 - [6]) * exp(-x / [8])) + [9])"

    ratio_1 = ROOT.TF1("ratio_1", f"{expr_ks} / ({expr_ks} + {expr_kl})", 0.0, 150.0)
    ratio_2 = ROOT.TF1("ratio_2", f"{expr_kl} / ({expr_ks} + {expr_kl})", 0.0, 150.0)

    for i in range(5):
        ratio_1.SetParameter(i, func.GetParameter(i))
        ratio_1.SetParameter(5 + i, func1.GetParameter(i))
        ratio_2.SetParameter(i, func.GetParameter(i))
        ratio_2.SetParameter(5 + i, func1.GetParameter(i))

    ratio_1.SetLineColor(ROOT.kGreen + 2)
    ratio_1.SetLineWidth(2)
    ratio_1.SetNpx(1000)

    ratio_2.SetLineColor(ROOT.kMagenta)
    ratio_2.SetLineWidth(2)
    ratio_2.SetNpx(1000)

    h_ratio0_res = hist_ratio_0.Clone("h_ratio0_res")
    h_ratio1_res = hist_ratio_1.Clone("h_ratio1_res")
    h_ratio0_res.SetDirectory(0)
    h_ratio1_res.SetDirectory(0)
    h_ratio0_res.Reset()
    h_ratio1_res.Reset()

    for b in range(1, hist_ratio_0.GetNbinsX() + 1):
        xc = hist_ratio_0.GetBinCenter(b)
        v0 = hist_ratio_0.GetBinContent(b)
        e0 = hist_ratio_0.GetBinError(b)
        v1 = hist_ratio_1.GetBinContent(b)
        e1 = hist_ratio_1.GetBinError(b)

        h_ratio0_res.SetBinContent(b, v0 - ratio_1.Eval(xc))
        h_ratio0_res.SetBinError(b, e0)

        h_ratio1_res.SetBinContent(b, v1 - ratio_2.Eval(xc))
        h_ratio1_res.SetBinError(b, e1)

legend = ROOT.TLegend(0.66, 0.82, 0.85, 0.9)
legend.AddEntry(hist_ratio_0, "K_{S}#rightarrow#pi^{+}#pi^{-}", "lp")
legend.AddEntry(hist_ratio_1, "K_{L}#rightarrow#pi^{+}#pi^{-}", "lp")

# 3. Rysowanie: góra = ratio (+ krzywe fitu, jeśli włączone), dół = reszty ratio (gdy oba fity włączone)
c = ROOT.TCanvas("c", "c", 700, 700)
if has_full_ratio_fit:
    pad1 = ROOT.TPad("pad1", "pad1", 0.0, 0.35, 1.0, 1.0)
    pad2 = ROOT.TPad("pad2", "pad2", 0.0, 0.0, 1.0, 0.35)
    pad1.SetBottomMargin(0.02)
    pad2.SetTopMargin(0.02)
    pad2.SetBottomMargin(0.28)
    pad1.Draw()
    pad2.Draw()
else:
    pad1 = ROOT.TPad("pad1", "pad1", 0.0, 0.0, 1.0, 1.0)
    pad2 = None
    pad1.Draw()

pad1.cd()
hist_ratio_0.SetLineColor(ROOT.kRed)
hist_ratio_1.SetLineColor(ROOT.kBlue)
hist_ratio_0.GetYaxis().SetRangeUser(0.0, 1.1)
hist_ratio_0.GetYaxis().SetTitle("Hist / Sum of hist")
hist_ratio_0.Draw()
hist_ratio_1.Draw("SAME")
if has_full_ratio_fit and ratio_1 is not None and ratio_2 is not None:
    ratio_1.Draw("SAME")
    ratio_2.Draw("SAME")
legend.Draw()

line_zero = None
if has_full_ratio_fit and pad2 is not None and h_ratio0_res is not None and h_ratio1_res is not None:
    pad2.cd()
    h_ratio0_res.SetLineColor(ROOT.kGreen + 2)
    h_ratio1_res.SetLineColor(ROOT.kMagenta)
    h_ratio0_res.GetYaxis().SetRangeUser(-0.15, 0.15)
    h_ratio0_res.GetYaxis().SetTitle("Ratio - Fit")
    h_ratio0_res.GetYaxis().SetTitleSize(0.08)
    h_ratio0_res.GetYaxis().SetLabelSize(0.07)
    h_ratio0_res.GetXaxis().SetTitleSize(0.08)
    h_ratio0_res.GetXaxis().SetLabelSize(0.07)
    h_ratio0_res.Draw("E")
    h_ratio1_res.Draw("E SAME")

    line_zero = ROOT.TLine(0.0, 0.0, 150.0, 0.0)
    line_zero.SetLineStyle(2)
    line_zero.Draw("SAME")

c.SaveAs(f"{output_dir}/decay_distance_ratio_{rebin_factor}_{channel[0]}_{channel[1]}_{enable_fits}.pdf")

KS fit: chi2/ndf = 9.40
KL fit: chi2/ndf = 2.31
****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      1058.01
Chi2                      =      2116.03
NDf                       =          225
Edm                       =  3.89894e-07
NCalls                    =          760
p0                        =  9.81411e+06   +/-   1.42221     
p1                        =     0.999216   +/-   1.45919e-05  	 (limited)
p2                        =      1.55813   +/-   0.00401379   	 (limited)
p3                        =      11.6947   +/-   0.0915929    	 (limited)
p4                        =      18.8415   +/-   0.480409     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      259.993
Chi2                      =      519.987
NDf                       =          225
Edm                       =   6.6919e-07
NCalls                    =          738
p0                        =      5914.78  

Info in <TCanvas::Print>: pdf file comparison/ks_pm_control_sample_kl_pm_control_sample/03e630a8_f63b399e/R_ch_R_KL/decay_distance_1_3pi0_pi+pi-pi+pi-_True.pdf has been created
Info in <TCanvas::Print>: pdf file comparison/ks_pm_control_sample_kl_pm_control_sample/03e630a8_f63b399e/R_ch_R_KL/decay_distance_ratio_1_3pi0_pi+pi-pi+pi-_True.pdf has been created


In [99]:
from array import array
import ROOT

ROOT.gROOT.SetBatch(True)

hist_type = ""
analysis_pairs = [("ks_pm_control_sample", "kl_pm_control_sample")]
analysis_hash_pairs = [("c97f42f6a601db1c64b593e259e04dec", "7b6c187f3e4b0390d51437281fd4292b")]
cut_hash = [("03e630a8", "f63b399e")]
var_pairs = [("R_ch", "R_KL")]
zoom_mode = "full"
channels = [("3pi0", "pi+pi-pi+pi-")]

enable_splines = True    # główny przełącznik spline'ów
enable_spline_ks = True
enable_spline_kl = True

run_spline_ks = enable_splines and enable_spline_ks
run_spline_kl = enable_splines and enable_spline_kl

ind = 0
tested_pair = analysis_pairs[ind]
analysis_hash = analysis_hash_pairs[ind]
cut_pair = cut_hash[ind]
var = var_pairs[ind]
channel = channels[ind]

output_dir = f"comparison/{tested_pair[0]}_{tested_pair[1]}/{cut_pair[0]}_{cut_pair[1]}/{var[0]}_{var[1]}"
os.makedirs(output_dir, exist_ok=True)

hist_files = []
histograms = []

for i in range(2):
    f = ROOT.TFile.Open(
        f"results/{tested_pair[i]}/{analysis_hash[i]}/img/{cut_pair[i]}/histograms.root",
        "READ",
    )
    hist_files.append(f)
    h = f.Get(f"{var[i]}/{var[i]}_{channel[i]}_{zoom_mode}")
    h.SetDirectory(0)
    histograms.append(h)

# 1. Ratio z histogramów
hist_sum = histograms[0].Clone("hist_sum")
hist_sum.SetDirectory(0)
hist_sum.Add(histograms[1])

hist_ratio_0 = histograms[0].Clone("hist_ratio_0")
hist_ratio_0.SetDirectory(0)
hist_ratio_0.Divide(hist_sum)

hist_ratio_1 = histograms[1].Clone("hist_ratio_1")
hist_ratio_1.SetDirectory(0)
hist_ratio_1.Divide(hist_sum)


def make_spline_from_hist(hist, name, xmin=0.0, xmax=150.0, step=1.0):
    xs, ys = array("d"), array("d")
    x = xmin
    while x <= xmax + 1e-9:
        b = hist.FindBin(x)
        if 1 <= b <= hist.GetNbinsX():
            xs.append(hist.GetBinCenter(b))
            ys.append(hist.GetBinContent(b))
        x += step
    graph = ROOT.TGraph(len(xs), xs, ys)
    graph.SetName(f"{name}_graph")
    spline = ROOT.TSpline3(name, graph)
    return spline, graph


# Krok węzłów spline'u w cm (np. 1.0 dla dokładniejszego zakrętu, 3.0-5.0 dla gładszego ogona)
spline_step = 5.0
spline_1 = None
g_spl_1 = None
spline_2 = None
g_spl_2 = None

if run_spline_ks:
    spline_1, g_spl_1 = make_spline_from_hist(
        hist_ratio_0, "spline_1", xmin=0.0, xmax=150.0, step=spline_step
    )
    spline_1.SetLineColor(ROOT.kGreen + 2)
    spline_1.SetLineWidth(2)

if run_spline_kl:
    spline_2, g_spl_2 = make_spline_from_hist(
        hist_ratio_1, "spline_2", xmin=0.0, xmax=150.0, step=spline_step
    )
    spline_2.SetLineColor(ROOT.kMagenta)
    spline_2.SetLineWidth(2)

# 2. Reszty na ratio względem spline'ów oraz odchyłka (spline_1 + spline_2) od 1.0
h_ratio0_res = None
h_ratio1_res = None
h_spline_sum_diff = None

if run_spline_ks and spline_1 is not None:
    h_ratio0_res = hist_ratio_0.Clone("h_ratio0_res")
    h_ratio0_res.SetDirectory(0)
    h_ratio0_res.Reset()

if run_spline_kl and spline_2 is not None:
    h_ratio1_res = hist_ratio_1.Clone("h_ratio1_res")
    h_ratio1_res.SetDirectory(0)
    h_ratio1_res.Reset()

has_both_splines = run_spline_ks and run_spline_kl and (spline_1 is not None) and (spline_2 is not None)
if has_both_splines:
    h_spline_sum_diff = hist_ratio_0.Clone("h_spline_sum_diff")
    h_spline_sum_diff.SetDirectory(0)
    h_spline_sum_diff.Reset()

sum_sq_0 = 0.0
sum_sq_1 = 0.0
sum_sq_norm = 0.0
max_abs_norm_diff = 0.0
x_max_norm_diff = 0.0
n_pts = 0

if run_spline_ks or run_spline_kl:
    for b in range(1, hist_ratio_0.GetNbinsX() + 1):
        xc = hist_ratio_0.GetBinCenter(b)
        v0 = hist_ratio_0.GetBinContent(b)
        e0 = hist_ratio_0.GetBinError(b)
        v1 = hist_ratio_1.GetBinContent(b)
        e1 = hist_ratio_1.GetBinError(b)

        s1 = spline_1.Eval(xc) if (run_spline_ks and spline_1 is not None) else None
        s2 = spline_2.Eval(xc) if (run_spline_kl and spline_2 is not None) else None

        if s1 is not None and h_ratio0_res is not None:
            r0 = v0 - s1
            h_ratio0_res.SetBinContent(b, r0)
            h_ratio0_res.SetBinError(b, e0)
            if 0.0 <= xc <= 150.0:
                sum_sq_0 += r0 * r0

        if s2 is not None and h_ratio1_res is not None:
            r1 = v1 - s2
            h_ratio1_res.SetBinContent(b, r1)
            h_ratio1_res.SetBinError(b, e1)
            if 0.0 <= xc <= 150.0:
                sum_sq_1 += r1 * r1

        if s1 is not None and s2 is not None and h_spline_sum_diff is not None:
            norm_diff = (s1 + s2) - 1.0
            h_spline_sum_diff.SetBinContent(b, norm_diff)
            h_spline_sum_diff.SetBinError(b, 0.0)
            if 0.0 <= xc <= 150.0:
                sum_sq_norm += norm_diff * norm_diff
                if abs(norm_diff) > max_abs_norm_diff:
                    max_abs_norm_diff = abs(norm_diff)
                    x_max_norm_diff = xc
                n_pts += 1

if run_spline_ks and n_pts > 0:
    print(f"Spline 1 RMS residual (0..150 cm): {(sum_sq_0 / n_pts) ** 0.5:.5f}")
if run_spline_kl and n_pts > 0:
    print(f"Spline 2 RMS residual (0..150 cm): {(sum_sq_1 / n_pts) ** 0.5:.5f}")
if has_both_splines and n_pts > 0:
    print(
        f"(spline_1 + spline_2 - 1) RMS (0..150 cm): {(sum_sq_norm / n_pts) ** 0.5:.6f}"
    )
    print(
        f"(spline_1 + spline_2 - 1) max abs: {max_abs_norm_diff:.6f} at x = {x_max_norm_diff:.2f} cm"
    )

# 3. Zapis spline'ów, grafów i histogramów ratio do pliku ROOT
out_path = "splines_and_ratios.root"
out_file = ROOT.TFile.Open(out_path, "RECREATE")
if run_spline_ks and spline_1 is not None and g_spl_1 is not None:
    spline_1.Write("spline_1")
    g_spl_1.Write("spline_1_graph")
if run_spline_kl and spline_2 is not None and g_spl_2 is not None:
    spline_2.Write("spline_2")
    g_spl_2.Write("spline_2_graph")
hist_ratio_0.Write("hist_ratio_0")
hist_ratio_1.Write("hist_ratio_1")
if h_ratio0_res is not None:
    h_ratio0_res.Write("h_ratio0_res")
if h_ratio1_res is not None:
    h_ratio1_res.Write("h_ratio1_res")
if h_spline_sum_diff is not None:
    h_spline_sum_diff.Write("h_spline_sum_diff")
out_file.Close()
print(f"Saved splines and ratio histograms to {out_path}")

legend = ROOT.TLegend(0.66, 0.82, 0.85, 0.9)
legend.AddEntry(hist_ratio_0, "K_{S}#rightarrow#pi^{+}#pi^{-}", "lp")
legend.AddEntry(hist_ratio_1, "K_{L}#rightarrow#pi^{+}#pi^{-}", "lp")

has_any_spline_res = (h_ratio0_res is not None) or (h_ratio1_res is not None)

# 4. Rysowanie: góra = ratio (+ spline'y, jeśli włączone), dół = reszty ratio (gdy włączone)
c = ROOT.TCanvas("c", "c", 700, 750)
if has_any_spline_res:
    pad1 = ROOT.TPad("pad1", "pad1", 0.0, 0.35, 1.0, 1.0)
    pad2 = ROOT.TPad("pad2", "pad2", 0.0, 0.0, 1.0, 0.35)
    pad1.SetBottomMargin(0.02)
    pad2.SetTopMargin(0.02)
    pad2.SetBottomMargin(0.28)
    pad1.Draw()
    pad2.Draw()
else:
    pad1 = ROOT.TPad("pad1", "pad1", 0.0, 0.0, 1.0, 1.0)
    pad2 = None
    pad1.Draw()

pad1.cd()
hist_ratio_0.SetLineColor(ROOT.kRed)
hist_ratio_1.SetLineColor(ROOT.kBlue)
hist_ratio_0.GetYaxis().SetRangeUser(0.0, 1.1)
hist_ratio_0.GetYaxis().SetTitle("Hist / Sum of hist")
hist_ratio_0.Draw()
hist_ratio_1.Draw("SAME")
if run_spline_ks and spline_1 is not None:
    spline_1.Draw("SAME")
if run_spline_kl and spline_2 is not None:
    spline_2.Draw("SAME")
legend.Draw()

line_zero = None
if has_any_spline_res and pad2 is not None:
    pad2.cd()
    first_res = True
    if h_ratio0_res is not None:
        h_ratio0_res.SetLineColor(ROOT.kGreen + 2)
        h_ratio0_res.GetYaxis().SetRangeUser(-0.15, 0.15)
        h_ratio0_res.GetYaxis().SetTitle("Ratio - Spline")
        h_ratio0_res.GetYaxis().SetTitleSize(0.08)
        h_ratio0_res.GetYaxis().SetLabelSize(0.07)
        h_ratio0_res.GetXaxis().SetTitleSize(0.08)
        h_ratio0_res.GetXaxis().SetLabelSize(0.07)
        h_ratio0_res.Draw("E")
        first_res = False

    if h_ratio1_res is not None:
        h_ratio1_res.SetLineColor(ROOT.kMagenta)
        if first_res:
            h_ratio1_res.GetYaxis().SetRangeUser(-0.15, 0.15)
            h_ratio1_res.GetYaxis().SetTitle("Ratio - Spline")
            h_ratio1_res.GetYaxis().SetTitleSize(0.08)
            h_ratio1_res.GetYaxis().SetLabelSize(0.07)
            h_ratio1_res.GetXaxis().SetTitleSize(0.08)
            h_ratio1_res.GetXaxis().SetLabelSize(0.07)
            h_ratio1_res.Draw("E")
        else:
            h_ratio1_res.Draw("E SAME")

    line_zero = ROOT.TLine(0.0, 0.0, 150.0, 0.0)
    line_zero.SetLineStyle(2)
    line_zero.Draw("SAME")

c.Draw()
c.SaveAs(f"{output_dir}/decay_distance_ratio_{rebin_factor}_{channel[0]}_{channel[1]}_splines_{enable_splines}.pdf")

Spline 1 RMS residual (0..150 cm): 0.01379
Spline 2 RMS residual (0..150 cm): 0.01379
(spline_1 + spline_2 - 1) RMS (0..150 cm): 0.000000
(spline_1 + spline_2 - 1) max abs: 0.000000 at x = 18.90 cm
Saved splines and ratio histograms to splines_and_ratios.root


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c
Info in <TCanvas::Print>: pdf file comparison/ks_pm_control_sample_kl_pm_control_sample/03e630a8_f63b399e/R_ch_R_KL/decay_distance_ratio_1_3pi0_pi+pi-pi+pi-_splines_True.pdf has been created


In [24]:
import ROOT

analysis_pairs = [("kl_pm_control_sample","kl_pm_control_sample")]
analysis_hash_pairs = [("7b6c187f3e4b0390d51437281fd4292b", "7b6c187f3e4b0390d51437281fd4292b")]
cut_hash= [("11f92bec", "f63b399e")]
var = "efficiency_MC/efficiency_Delta_t_fit_MC_subtract_bkg"

ind = 0
tested_pair = analysis_pairs[ind]
analysis_hash = analysis_hash_pairs[ind]
cut_pair = cut_hash[ind]

hist_files = []
histograms = []

for i in range(2):
    hist_files.append(ROOT.TFile.Open(f"results/{tested_pair[i]}/{analysis_hash[i]}/img/{cut_pair[i]}/efficiency_purity_plots/eff_pur_corr_factor.root", "READ"))

    histograms.append(hist_files[i].Get(var))

############ Preparation #######################

hist_res = []

for i in range(2):
  hist_res.append(histograms[i])


hist_div = hist_res[1].Clone("hist_div")
hist_div.Divide(hist_res[0])


legend = ROOT.TLegend(0.7, 0.8, 0.9, 0.9)
legend.AddEntry(hist_res[0], "Before cut P_{miss} < 10 MeV/c", "l")
legend.AddEntry(hist_res[1], "After cut P_{miss} < 10 MeV/c", "l")

############ Draw histograms ####################
c = ROOT.TCanvas("c", "c", 600, 600)

p1 = ROOT.TPad("p1", "p1", 0, 0.3, 1, 1)
c.cd()
p1.SetBottomMargin(0.02)
p1.Draw()

p2 = ROOT.TPad("p2", "p2", 0, 0, 1, 0.3)
c.cd()
p2.SetTopMargin(0.02)
p2.SetBottomMargin(0.3)
p2.Draw()

p1.cd()

hist_res[0].SetLineColor(ROOT.kRed)
hist_res[1].SetLineColor(ROOT.kBlue)

hist_res[0].SetMarkerColor(ROOT.kRed)
hist_res[1].SetMarkerColor(ROOT.kBlue)

hist_res[0].Draw()
hist_res[1].Draw("SAME")
legend.Draw()

p2.cd()
hist_div.GetYaxis().SetTitle("Ratio After/Before Cut")
hist_div.GetYaxis().SetNdivisions(505)
hist_div.GetYaxis().SetRangeUser(0.8, 1.1)
hist_div.SetLineColor(ROOT.kBlack)
hist_div.SetMarkerColor(ROOT.kBlack)
hist_div.Draw()

c.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c


In [62]:
import ROOT

hist_type = "ratio"
analysis_pairs = [("kl_pm_control_sample","kl_pm_control_sample")]
analysis_hash_pairs = [("7b6c187f3e4b0390d51437281fd4292b", "7b6c187f3e4b0390d51437281fd4292b")]
cut_hash= [("f63b399e", "11f92bec")]
var_pairs = [("Delta_t_fit","Delta_t_fit")]
zoom_mode = "full"
channels = [("Other", "Other")]

ind = 0
tested_pair = analysis_pairs[ind]
analysis_hash = analysis_hash_pairs[ind]
cut_pair = cut_hash[ind]
var = var_pairs[ind]
channel = channels[ind]

hist_files = []
histograms = []

for i in range(2):
    hist_files.append(ROOT.TFile.Open(f"results/{tested_pair[i]}/{analysis_hash[i]}/img/{cut_pair[i]}/histograms.root", "READ"))

    histograms.append(hist_files[i].Get(f"{var[i]}/{var[i]}_{channel[i]}_{zoom_mode}"))

############ Preparation #######################

hist_res = []
if hist_type == "ratio":

  for i in range(2):
    hist = histograms[i].Clone(f"hist_ratio_{i}")
    hist.Scale(1.0 / histograms[i].Integral())
    hist_res.append(hist)

else:
  for i in range(2):
    hist_res.append(histograms[i])

hist_sub = histograms[0].Clone("hist_sub")
hist_sub.Add(histograms[1], -1)
hist_sub.Divide(histograms[0])
print("hist_sub created")

############ Draw histograms ####################
ROOT.gStyle.SetOptStat(0)

c = ROOT.TCanvas("c", "c", 600, 600)

legend = ROOT.TLegend(0.7, 0.7, 0.9, 0.9)
legend.AddEntry(hist_res[0], "Hist after cut", "l")
legend.AddEntry(hist_res[1], "Hist before cut", "l")

p1 = ROOT.TPad("p1", "p1", 0, 0.3, 1, 1)
c.cd()
p1.SetBottomMargin(0.02)
p1.Draw()

p2 = ROOT.TPad("p2", "p2", 0, 0, 1, 0.3)
c.cd()
p2.SetTopMargin(0.02)
p2.SetBottomMargin(0.3)
p2.Draw()

p1.cd()
hist_res[0].SetLineColor(ROOT.kRed)
hist_res[1].SetLineColor(ROOT.kBlue)
# hist_res[0].GetXaxis().SetRangeUser(0.0, 0.0)

if hist_type == "ratio":
    # hist_res[0].GetYaxis().SetRangeUser(0.0, 1.1)
    hist_res[0].GetYaxis().SetTitle("Normalized counts")

hist_res[0].GetXaxis().SetRangeUser(-10, 100)
hist_res[0].Draw()
hist_res[1].Draw("SAME")
legend.Draw()


p2.cd()

hist_sub.GetXaxis().SetRangeUser(-10, 100)
hist_sub.GetYaxis().SetTitle("(Hist_0 - Hist_1) / Hist_0")

hist_sub.Draw()
c.Draw()

hist_sub created


In [32]:
import ROOT

hist_type = "ratio"
analysis_name = "signal_analysis_cut_checks"
analysis_hash = "c97f42f6a601db1c64b593e259e04dec"
cut_hash = "9138eb05"
var = "Time_Ne_fit_Time_Ne_tri_fit"
zoom_mode = "full"
channel = "MC sum"

hist_file = ROOT.TFile.Open(f"results/{analysis_name}/{analysis_hash}/signal_analysis_chann_div_2D_variables.root", "READ")

histogram = hist_file.Get(f"{cut_hash}/{var}/{var}_{zoom_mode}_{channel}")

############ Preparation #######################

histogram.SetDirectory(0)


import math
import ROOT

# Zakładając, że masz już wczytany histogram h2 (np. TH2D lub TH2F)
h2 = histogram
offset = 80.0
n_above = 0.0
n_below = 0.0
err2_above = 0.0
err2_below = 0.0

xaxis = h2.GetXaxis()
yaxis = h2.GetYaxis()

for ix in range(1, h2.GetNbinsX() + 1):
    x = xaxis.GetBinCenter(ix)
    for iy in range(1, h2.GetNbinsY() + 1):
        y = yaxis.GetBinCenter(iy)
        val = h2.GetBinContent(ix, iy)
        err = h2.GetBinError(ix, iy)

        if y > x + offset:
            n_above += val
            err2_above += err * err
        elif y < x + offset:
            n_below += val
            err2_below += err * err

err_above = math.sqrt(err2_above)
err_below = math.sqrt(err2_below)

print(f"Powyżej linii: {n_above} +/- {err_above}")
print(f"Poniżej linii: {n_below} +/- {err_below}")


line = ROOT.TLine(0, offset, histogram.GetNbinsX(), histogram.GetNbinsX() + offset)
line.SetLineColor(ROOT.kRed)

canva = ROOT.TCanvas("canva", "canva", 600, 600)
histogram.Draw()
line.Draw("same")
canva.Update()
canva.Draw()
canva.SaveAs("histogram_with_line.png")

Powyżej linii: 17040.081453863928 +/- 138.83120667386362
Poniżej linii: 162600.72149481333 +/- 428.85699619036


Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva
Info in <TCanvas::Print>: png file histogram_with_line.png has been created


In [61]:
from array import array
import math
import ROOT

hist_type = "ratio"
analysis_name = "signal_analysis_cut_checks"
analysis_hash = "c97f42f6a601db1c64b593e259e04dec"
cut_hash = "9138eb05"
var = "Delta_t_MC_Delta_t_fit_resolution"
zoom_mode = "full"
channel = "Signal"

# Opcje rebinowania (1 = bez rebinu; np. rebin_x = 5 łączy po 5 binów wzdłuż osi X)
rebin_x = 5
rebin_y = 1

# Szersze okno do wstępnego wyznaczenia centrum piku z fitu
pre_fit_half_width = 10.0

# Półszerokość właściwego okna dopasowania wokół wyznaczonego centrum piku
fit_half_width = 7.0

hist_file = ROOT.TFile.Open(
    f"results/{analysis_name}/{analysis_hash}/signal_analysis_chann_div_2D_truth_variables.root",
    "READ",
)

histogram = hist_file.Get(f"{cut_hash}/{var}/{var}_{zoom_mode}_{channel}")

############ Preparation #######################

histogram.SetDirectory(0)

work_hist = histogram.Clone("hist_rebinned")
work_hist.SetDirectory(0)
if rebin_x > 1 or rebin_y > 1:
    work_hist.Rebin2D(rebin_x, rebin_y)

canva = ROOT.TCanvas("canva", "canva", 600, 600)
work_hist.Draw("COLZ")
canva.Draw()
canva.SaveAs("histogram_with_line.png")

############ Profil w osi Y #######################

prof_y = work_hist.ProfileX("_py")
prof_y.SetDirectory(0)
prof_y.SetTitle(f";{work_hist.GetXaxis().GetTitle()};#mu({str(work_hist.GetYaxis().GetTitle())})")
prof_y.SetLineColor(ROOT.kRed)
prof_y.SetLineWidth(2)

canva_prof_y = ROOT.TCanvas("canva_prof_y", "Profil w osi Y", 600, 600)
prof_y.Draw()
canva_prof_y.Draw()
canva_prof_y.SaveAs("profile_y.png")

###################### Odchylenie standardowe w osi Y #######################

def is_valid_candidate(cand, max_sigma_err=5.0):
    if cand is None:
        return False
    _, _, sigma, sigma_err, _ = cand
    return (
        math.isfinite(sigma)
        and math.isfinite(sigma_err)
        and sigma > 0.0
        and sigma_err > 0.0
        and sigma_err < max_sigma_err
    )


def stability_score(cand, prefer_1gaus_margin=3.0, sigma_weight=10.0):
    """
    Mniejsza wartość = lepszy kandydat (bardziej stabilne sigma oraz mniejsze sigma).
    Łączy błąd względny sigma_err / sigma z karą proporcjonalną do sigma.
    """
    _, _, sigma, sigma_err, label = cand
    rel_err = sigma_err / sigma
    if label == "1xGauss":
        rel_err *= prefer_1gaus_margin
    return rel_err * (1.0 + sigma_weight * sigma)


def get_histogram_median(h):
    prob = array("d", [0.5])
    quant = array("d", [0.0])
    h.GetQuantiles(1, quant, prob)
    return quant[0]


delta_t = []
mu_list = []
mu_err_list = []
sigma_list = []
sigma_err_list = []

for i in range(1, work_hist.GetNbinsX() + 1):
    proj_y = work_hist.ProjectionY("_px", i, i)
    proj_y.SetDirectory(0)

    if proj_y.GetEntries() < 20 or proj_y.GetRMS() <= 0 or proj_y.GetMaximum() <= 0:
        continue

    yaxis = proj_y.GetXaxis()
    axis_min = yaxis.GetXmin()
    axis_max = yaxis.GetXmax()

    # Krok 1: Wstępne wyznaczenie centrum piku z fitu Gaussa w szerszym oknie
    seed_center = yaxis.GetBinCenter(proj_y.GetMaximumBin())
    if not math.isfinite(seed_center):
        seed_center = get_histogram_median(proj_y)
    if not math.isfinite(seed_center):
        continue

    pre_min = max(axis_min, seed_center - pre_fit_half_width)
    pre_max = min(axis_max, seed_center + pre_fit_half_width)
    if pre_max <= pre_min:
        continue

    bin_pre_lo = yaxis.FindBin(pre_min)
    bin_pre_hi = yaxis.FindBin(pre_max)
    yaxis.SetRange(bin_pre_lo, bin_pre_hi)
    pre_mean = proj_y.GetMean()
    pre_rms = proj_y.GetRMS()
    pre_max_val = proj_y.GetMaximum()
    yaxis.SetRange(0, 0)

    if pre_rms <= 0 or pre_max_val <= 0:
        continue

    pre_func = ROOT.TF1("pre_func", "gaus", pre_min, pre_max)
    pre_func.SetParameters(
        pre_max_val,
        pre_mean if math.isfinite(pre_mean) else seed_center,
        pre_rms,
    )
    pre_res = proj_y.Fit("pre_func", "SRQR")

    if pre_res and not pre_res.Status() and math.isfinite(pre_res.Parameter(1)):
        peak_center = pre_res.Parameter(1)
    else:
        peak_center = pre_mean if math.isfinite(pre_mean) else seed_center

    if not (axis_min <= peak_center <= axis_max):
        peak_center = seed_center

    # Krok 2: Właściwe okno wokół centrum z fitu
    fit_min = max(axis_min, peak_center - fit_half_width)
    fit_max = min(axis_max, peak_center + fit_half_width)
    if fit_max <= fit_min:
        continue

    bin_lo = yaxis.FindBin(fit_min)
    bin_hi = yaxis.FindBin(fit_max)
    yaxis.SetRange(bin_lo, bin_hi)
    local_mean = proj_y.GetMean()
    local_rms = proj_y.GetRMS()
    local_max = proj_y.GetMaximum()
    yaxis.SetRange(0, 0)

    if local_rms <= 0 or local_max <= 0:
        continue

    mean_init = local_mean if math.isfinite(local_mean) else peak_center

    # 1. Fit pojedynczym Gaussem
    func_1g = ROOT.TF1("func_1g", "gaus", fit_min, fit_max)
    func_1g.SetParameters(local_max, mean_init, local_rms)
    res_1g = proj_y.Fit("func_1g", "SRQR")

    cand_1g = None
    if res_1g and not res_1g.Status():
        cand_1g = (
            res_1g.Parameter(1),
            res_1g.ParError(1),
            res_1g.Parameter(2),
            res_1g.ParError(2),
            "1xGauss",
        )
        if not is_valid_candidate(cand_1g):
            cand_1g = None

    # 2. Fit podwójnym Gaussem (ROOT.double_gaussian_cpp)
    func_2g = ROOT.TF1("func_2g", ROOT.double_gaussian_cpp, fit_min, fit_max, 5)
    func_2g.SetParameters(
        local_max,
        mean_init,
        local_rms * 0.5,
        local_rms * 1.5,
        0.7,
    )
    func_2g.SetParLimits(0, 0, local_max * 50.0)
    func_2g.SetParLimits(1, fit_min, fit_max)
    func_2g.SetParLimits(2, 0, local_rms * 2.0)
    func_2g.SetParLimits(3, 0, local_rms * 4.0)
    func_2g.SetParLimits(4, 0, 1)
    res_2g = proj_y.Fit("func_2g", "SRQR")

    cand_2g = None
    if res_2g and not res_2g.Status():
        if res_2g.Parameter(2) < res_2g.Parameter(3):
            s2 = res_2g.Parameter(2)
            s2_err = res_2g.ParError(2)
        else:
            s2 = res_2g.Parameter(3)
            s2_err = res_2g.ParError(3)
        cand_2g = (
            res_2g.Parameter(1),
            res_2g.ParError(1),
            s2,
            s2_err,
            "2xGauss",
        )
        if not is_valid_candidate(cand_2g):
            cand_2g = None

    # Wybór bardziej stabilnego i mniejszego sigma
    candidates = [c for c in (cand_1g, cand_2g) if c is not None]
    if not candidates:
        continue

    best = min(candidates, key=stability_score)
    mu, mu_err, sigma, sigma_err, chosen_label = best

    delta_t.append(work_hist.GetXaxis().GetBinCenter(i))
    mu_list.append(mu)
    mu_err_list.append(mu_err)
    sigma_list.append(sigma)
    sigma_err_list.append(sigma_err)

graph_mu = ROOT.TGraphErrors(
    len(delta_t),
    array("d", delta_t),
    array("d", mu_list),
    array("d", [0] * len(delta_t)),
    array("d", mu_err_list),
)
graph_mu.SetTitle(
    f";{work_hist.GetXaxis().GetTitle()};"
    f"#mu({str(work_hist.GetYaxis().GetTitle()).removesuffix(' [#tau_{S}]')}) [#tau_{{S}}]"
)
graph_mu.SetMarkerStyle(20)
graph_mu.SetMarkerSize(0.7)
graph_mu.SetMarkerColor(ROOT.kBlue)
graph_mu.SetLineColor(ROOT.kBlue)
graph_mu.SetLineWidth(1)

canva_mu = ROOT.TCanvas("canva_mu", "Mean vs Delta_t", 600, 600)
canva_mu.SetLeftMargin(0.18)
graph_mu.Draw("APE1")
canva_mu.Draw()
canva_mu.SaveAs("mean_vs_delta_t.png")

graph_sigma = ROOT.TGraphErrors(
    len(delta_t),
    array("d", delta_t),
    array("d", sigma_list),
    array("d", [0] * len(delta_t)),
    array("d", sigma_err_list),
)
graph_sigma.SetTitle(
    f";{work_hist.GetXaxis().GetTitle()};"
    f"#sigma({str(work_hist.GetYaxis().GetTitle()).removesuffix(' [#tau_{S}]')}) [#tau_{{S}}]"
)
graph_sigma.SetMarkerStyle(20)
graph_sigma.SetMarkerSize(0.7)
graph_sigma.SetMarkerColor(ROOT.kRed)
graph_sigma.SetLineColor(ROOT.kRed)
graph_sigma.SetLineWidth(1)
graph_sigma.GetYaxis().SetRangeUser(0.0, 6.0)

canva_sigma = ROOT.TCanvas("canva_sigma", "StdDev vs Delta_t", 600, 600)
canva_sigma.SetLeftMargin(0.18)
graph_sigma.Fit("pol0", "")
graph_sigma.Draw("APE1")
canva_sigma.Draw()
canva_sigma.SaveAs("stddev_vs_delta_t.png")

###################### Projekcje dla -10 i +10 binów wokół 0 #######################

nbins_x = work_hist.GetNbinsX()
bin_zero = work_hist.GetXaxis().FindBin(0.0)
bin_minus10 = max(1, min(nbins_x, bin_zero - 10))
bin_plus10 = max(1, min(nbins_x, bin_zero + 10))

x_val_minus10 = work_hist.GetXaxis().GetBinCenter(bin_minus10)
x_val_plus10 = work_hist.GetXaxis().GetBinCenter(bin_plus10)

proj_minus10 = work_hist.ProjectionY("_proj_bin_minus10", bin_minus10, bin_minus10)
proj_minus10.SetDirectory(0)
proj_minus10.SetTitle(f";{work_hist.GetYaxis().GetTitle()};Events")
proj_minus10.SetLineColor(ROOT.kBlue + 1)
proj_minus10.SetLineWidth(2)

proj_plus10 = work_hist.ProjectionY("_proj_bin_plus10", bin_plus10, bin_plus10)
proj_plus10.SetDirectory(0)
proj_plus10.SetTitle(f";{work_hist.GetYaxis().GetTitle()};Events")
proj_plus10.SetLineColor(ROOT.kRed + 1)
proj_plus10.SetLineWidth(2)

max_proj_y = max(proj_minus10.GetMaximum(), proj_plus10.GetMaximum())
if max_proj_y > 0:
    proj_minus10.SetMaximum(max_proj_y * 1.15)
    proj_plus10.SetMaximum(max_proj_y * 1.15)

canva_proj_0 = ROOT.TCanvas("canva_proj_0", "Projections -10 and +10 bins around 0", 650, 600)
canva_proj_0.SetLeftMargin(0.16)
proj_minus10.Draw("HIST")
proj_plus10.Draw("HIST SAME")

leg_0 = ROOT.TLegend(0.55, 0.72, 0.88, 0.88)
leg_0.SetBorderSize(0)
leg_0.SetFillStyle(0)
leg_0.AddEntry(proj_minus10, f"bin -10 (x = {x_val_minus10:.1f})", "l")
leg_0.AddEntry(proj_plus10, f"bin +10 (x = {x_val_plus10:.1f})", "l")
leg_0.Draw()

canva_proj_0.Draw()
canva_proj_0.SaveAs("proj_minus10_plus10_around_0.png")

****************************************
Minimizer is Linear / Migrad
Chi2                      =      1261.44
NDf                       =           79
p0                        =      1.72728   +/-   0.00522497  


Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva
Info in <TCanvas::Print>: png file histogram_with_line.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva_prof_y
Info in <TCanvas::Print>: png file profile_y.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva_mu
Info in <TCanvas::Print>: png file mean_vs_delta_t.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva_sigma
Info in <TCanvas::Print>: png file stddev_vs_delta_t.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva_proj_0
Info in <TCanvas::Print>: png file proj_minus10_plus10_around_0.png has been created
